# Duckweed Analysis Pipeline
### Unified Analysis for Petri Dish & Microfluidics

**How to use:**
1. Set `DATASET_TYPE = "petri_dish"` or `"microfluidics"` in Section 1.2
2. Run cells in order

**Structure:**
- **Part 1:** Setup (imports, config, model)
- **Part 2:** Single Image Demo
- **Part 3:** Full Pipeline (segmentation + tracking)
- **Part 4:** Cluster Analysis
- **Part 5:** Data Processing
- **Part 6:** Visualizations
- **Part 7:** Statistical Analysis
- **Part 8:** Summary

## Overview
This notebook generates all figures and analysis for the duckweed tracking paper.

## Story Structure (Order of Presentation)
1. **Segmentation** - Show the model can identify individual duckweed
2. **Tracking** - Show we can follow individuals over time
3. **Single Cluster Analysis** - Demonstrate metrics on one cluster (builds trust)
4. **Population Dynamics** - Number of individuals over time (step function)
5. **All Clusters** - Extend analysis to entire petri dish
6. **Generations & Lineage** - Family relationships (LAST)

## Key Principle
> "Every plot line must be backed by visual proof" - Prof. Sam

Each figure should have corresponding raw images to build reader trust.

---
# PART 1: SETUP
---

## 1.1 Imports
Load all required libraries.

In [ ]:
# Core libraries
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# Image processing
import cv2
from PIL import Image
from skimage import measure
from skimage.measure import regionprops, regionprops_table
from skimage.color import label2rgb
from skimage.transform import resize

# Visualization
import matplotlib.pyplot as plt
from matplotlib import gridspec
import networkx as nx

# Data augmentation & preprocessing
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Tracking
import btrack
from btrack.utils import segmentation_to_objects
from scipy.spatial import cKDTree

# Custom modules (from this project)
import sys
# UNet model class is in the same directory on the server
import unet_model_class
# import unet_model_v2  # Not needed for boundary model
# from distance_map_to_instances import distance_map_to_instances  # Not needed for boundary model

print("All imports successful!")


## 1.2 Configuration
Set `DATASET_TYPE`, paths, and parameters. **This is the main control cell.**

In [ ]:
# ============================================
# CONFIGURATION - Modify these as needed
# ============================================
import os
from pathlib import Path
import cv2

# When running from app, config was loaded in previous cell; else set defaults here
if not os.environ.get("DUCKWEED_CONFIG"):
    # ==========================================
    # DATASET SELECTION - Choose one:
    # ==========================================
    DATASET_TYPE = "petri_dish"  # Options: "petri_dish" or "microfluidics"

    # ==========================================
    # SERVER BASE PATH
    # ==========================================
    BASE_PATH = "/Users/smdoliveira/Desktop/Deepweed/paper_analysis"

    # ==========================================
    # PATHS & PARAMETERS - Set per dataset
    # ==========================================
    if DATASET_TYPE == "petri_dish":
        # Petri Dish settings
        IMAGE_DIRECTORY = BASE_PATH + "/image_data/petri_dish/"
        VIDEO_PATH = None  # Not used for petri dish
        OUTPUT_DIR = BASE_PATH + "/paper_analysis_output/petri_dish/"
        MODEL_LOCATION = BASE_PATH + "/model/best_model_0707.pt"
        BTRACK_CONFIG = BASE_PATH + "/cell_config.json"

        # Time: 5 minutes per frame
        MINUTES_PER_FRAME = 5

        # Segmentation parameters (tuned for petri dish - larger duckweed)
        THRESHOLD = 0.2
        MIN_DISTANCE = 7
        MIN_AREA = 30
        CLUSTER_DISTANCE = 150
        N_CLUSTERS = 6  # Number of petri dishes in the image (2x3 grid)

        # Input type flag
        USE_VIDEO = False

        # Segmentation method: Watershed for petri dish (plants can touch)
        USE_WATERSHED = False

    elif DATASET_TYPE == "microfluidics":
        # Microfluidics settings
        IMAGE_DIRECTORY = None  # Not used for microfluidics
        VIDEO_PATH = BASE_PATH + "/image_data/Microfluidics_video_data/duckweed_25_0504_multiple.avi"
        OUTPUT_DIR = BASE_PATH + "/paper_analysis_output/microfluidics/"
        MODEL_LOCATION = BASE_PATH + "/model/best_model_0707.pt"
        BTRACK_CONFIG = BASE_PATH + "/cell_config.json"

        # Time: 20 minutes per frame
        MINUTES_PER_FRAME = 20

        # Segmentation parameters (tuned for microfluidics - smaller wells)
        THRESHOLD = 0.2
        MIN_DISTANCE = 5
        MIN_AREA = 20
        CLUSTER_DISTANCE = 40
        N_CLUSTERS = None  # Auto-detect for microfluidics (uses CLUSTER_DISTANCE)

        # Input type flag
        USE_VIDEO = True

        # Segmentation method: Simple threshold for microfluidics (plants are isolated)
        USE_WATERSHED = False

    else:
        raise ValueError(f"Unknown DATASET_TYPE: {DATASET_TYPE}. Use 'petri_dish' or 'microfluidics'")

# Processing parameters
FRAME_LIMIT = None  # Process ALL frames (set to integer to limit)

# ==========================================
# Count frames and display info
# ==========================================
if USE_VIDEO:
    cap = cv2.VideoCapture(VIDEO_PATH)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    input_source = VIDEO_PATH
else:
    all_image_files = sorted(list(Path(IMAGE_DIRECTORY).glob('*.jpeg')) + list(Path(IMAGE_DIRECTORY).glob('*.png')))
    total_frames = len(all_image_files)
    input_source = IMAGE_DIRECTORY

frames_to_process = total_frames if FRAME_LIMIT is None else min(FRAME_LIMIT, total_frames)
total_hours = (frames_to_process * MINUTES_PER_FRAME) / 60

print("=" * 50)
print(f"DATASET: {DATASET_TYPE.upper().replace('_', ' ')}")
print("=" * 50)
if USE_VIDEO:
    print(f"Video: {VIDEO_PATH}")
else:
    print(f"Images: {IMAGE_DIRECTORY}")
print(f"Model: {MODEL_LOCATION}")
print(f"Output: {OUTPUT_DIR}")
print()
print(f"Frames: {total_frames} available")
print(f"Processing: {frames_to_process} frames ({total_hours:.1f} hours)")
print(f"Segmentation: {'Watershed' if USE_WATERSHED else 'Simple threshold'}")

# Create output directory if needed
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# VISUALIZATION PARAMETERS (auto-set based on dataset)
# ==========================================
if DATASET_TYPE == "microfluidics":
    VIZ_LABEL_FONTSIZE = 8
    VIZ_TITLE_FONTSIZE = 14
    VIZ_AXIS_FONTSIZE = 10
    VIZ_LEGEND_FONTSIZE = 7
    VIZ_SCATTER_SIZE = 20
    VIZ_FIG_SIZE = (10, 14)
else:  # petri_dish
    VIZ_LABEL_FONTSIZE = 16
    VIZ_TITLE_FONTSIZE = 24
    VIZ_AXIS_FONTSIZE = 14
    VIZ_LEGEND_FONTSIZE = 10
    VIZ_SCATTER_SIZE = 50
    VIZ_FIG_SIZE = (14, 10)

print(f"Viz settings: labels={VIZ_LABEL_FONTSIZE}pt, titles={VIZ_TITLE_FONTSIZE}pt")


## 1.3 Load Model
Load the trained U-Net segmentation model.

## 1.4 Checkpoint (Save/Load)
Save or restore tracking data to avoid re-running the pipeline.

In [ ]:
# ============================================
# SAVE CHECKPOINT - Run AFTER tracking completes
# ============================================
import pickle

checkpoint_path = OUTPUT_DIR + 'tracking_checkpoint.pkl'

# Check if data exists
has_unified = 'unified_df' in dir() and unified_df is not None and len(unified_df) > 0
has_lineage = 'lineage_df' in dir() and lineage_df is not None
has_extracted = 'extracted_df' in dir() and extracted_df is not None and len(extracted_df) > 0

if not has_unified and not has_extracted:
    print("⚠️  WARNING: No tracking data found!")
    print("   unified_df and extracted_df are empty or don't exist.")
    print("")
    print("   Run tracking cells first, then come back here to save.")
else:
    checkpoint_data = {
        'unified_df': unified_df if has_unified else None,
        'lineage_df': lineage_df if has_lineage else None,
        'extracted_df': extracted_df if has_extracted else None,
        'population_per_frame_filtered': population_per_frame_filtered if 'population_per_frame_filtered' in dir() else None,
        'DISH_ROI': DISH_ROI if 'DISH_ROI' in dir() else None,
        'MINUTES_PER_FRAME': MINUTES_PER_FRAME if 'MINUTES_PER_FRAME' in dir() else 5,
    }
    
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint_data, f)
    
    print(f"✓ Checkpoint saved to: {checkpoint_path}")
    print(f"  • unified_df: {len(unified_df)} rows" if has_unified else "  • unified_df: None")
    print(f"  • lineage_df: {len(lineage_df)} rows" if has_lineage else "  • lineage_df: None")
    print(f"  • extracted_df: {len(extracted_df)} rows" if has_extracted else "  • extracted_df: None")


In [ ]:
# ============================================
# LOAD CHECKPOINT (if available)
# ============================================
# After loading, SKIP to Part 3 (Analysis) cells!
# Do NOT run cells 8-9 (tracking pipeline) - they need intermediate variables

import pickle
import os

checkpoint_path = OUTPUT_DIR + 'tracking_checkpoint.pkl'

if os.path.exists(checkpoint_path):
    print("Found checkpoint! Loading...")
    
    with open(checkpoint_path, 'rb') as f:
        checkpoint_data = pickle.load(f)
    
    # Restore variables
    unified_df = checkpoint_data.get('unified_df')
    lineage_df = checkpoint_data.get('lineage_df')
    extracted_df = checkpoint_data.get('extracted_df')
    population_per_frame_filtered = checkpoint_data.get('population_per_frame_filtered')
    extracted_df_filtered = checkpoint_data.get('extracted_df_filtered')
    DISH_ROI = checkpoint_data.get('DISH_ROI')
    MINUTES_PER_FRAME = checkpoint_data.get('MINUTES_PER_FRAME', 5)
    
    # Restore paths if saved
    if checkpoint_data.get('IMAGE_DIRECTORY'):
        IMAGE_DIRECTORY = checkpoint_data['IMAGE_DIRECTORY']
    if checkpoint_data.get('OUTPUT_DIR'):
        OUTPUT_DIR = checkpoint_data['OUTPUT_DIR']
    
    has_data = unified_df is not None and len(unified_df) > 0
    
    if has_data:
        print("✓ Checkpoint loaded successfully!")
        print(f"  • unified_df: {len(unified_df)} rows")
        print(f"  • lineage_df: {len(lineage_df)} rows" if lineage_df is not None else "  • lineage_df: None")
        print(f"  • extracted_df: {len(extracted_df)} rows" if extracted_df is not None else "  • extracted_df: None")
        print(f"  • MINUTES_PER_FRAME: {MINUTES_PER_FRAME}")
        print("")
        print("=" * 50)
        print("⚠️  IMPORTANT: Skip to PART 3 (Analysis) cells!")
        print("   Do NOT run tracking cells - they need")
        print("   intermediate variables that aren't saved.")
        print("=" * 50)
    else:
        print("⚠️  Checkpoint exists but data is empty!")
        print("   Run tracking first, then save again.")
    
else:
    print("No checkpoint found.")
    print(f"Expected: {checkpoint_path}")
    print("")
    print("Run the full tracking pipeline (Section 8),")
    print("then run the SAVE cell to create checkpoint.")


## 1.3b Load Boundary Instance Model (NEW)
Load the 3-class boundary U-Net trained on hand-labelled instance masks.
Predicts: background (0), frond body (1), boundary (2).
Instances = connected components of body class.

In [ ]:
# ============================================
# LOAD SEGMENTATION MODEL
# ============================================
# Loads the correct model based on DATASET_TYPE:
#   - petri_dish:    3-class boundary UNet (bg/body/boundary) → connected components
#   - microfluidics: 1-class distance UNet → thresholded binary mask

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DATASET_TYPE == "petri_dish":
    # --- PETRI DISH: 3-class boundary model ---
    BOUNDARY_MODEL_PATH = BASE_PATH + "/model/best_instance_unet.pt"
    
    boundary_model = unet_model_class.UNet(n_channels=3, n_classes=3).to(device)
    ckpt_boundary = torch.load(BOUNDARY_MODEL_PATH, map_location=device, weights_only=False)
    boundary_model.load_state_dict(ckpt_boundary['model_state_dict'])
    boundary_model.eval()
    
    print(f"PETRI DISH model loaded: {BOUNDARY_MODEL_PATH}")
    print(f"  Architecture: 3-class boundary UNet (bg / body / boundary)")
    print(f"  Trained epoch: {ckpt_boundary.get('epoch', '?')}")
    print(f"  Best FG mIoU: {ckpt_boundary.get('best_fg_miou', '?'):.4f}")

elif DATASET_TYPE == "microfluidics":
    # --- MICROFLUIDICS: 1-class distance model ---
    # Uses MODEL_LOCATION from config cell
    
    boundary_model = unet_model_class.UNet(n_channels=3, n_classes=1).to(device)
    ckpt_boundary = torch.load(MODEL_LOCATION, map_location=device, weights_only=False)
    boundary_model.load_state_dict(ckpt_boundary['model_state_dict'])
    boundary_model.eval()
    
    print(f"MICROFLUIDICS model loaded: {MODEL_LOCATION}")
    print(f"  Architecture: 1-class distance UNet")
    print(f"  Trained epoch: {ckpt_boundary.get('epoch', '?')}")
    print(f"  Best val dice: {ckpt_boundary.get('best_val_dice', '?'):.4f}")

else:
    raise ValueError(f"Unknown DATASET_TYPE: {DATASET_TYPE}")


def segment_boundary_model(image_rgb, min_area=30):
    """
    Segment using the loaded model (works for both dataset types).
    
    For petri_dish:  3-class output → body class (1) → connected components
    For microfluidics: 1-class distance map → threshold → connected components
    
    Args:
        image_rgb: RGB numpy array (original size)
        min_area: minimum frond area in pixels
    
    Returns:
        instance_mask: HxW int array (0=bg, 1,2,3...=fronds)
        num_instances: number of fronds detected
    """
    original_shape = image_rgb.shape[:2]
    
    # Preprocess: resize to 256x256, normalize
    img = cv2.resize(image_rgb, (256, 256))
    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    input_tensor = torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).float().to(device)
    
    # Predict
    with torch.no_grad():
        output = boundary_model(input_tensor)
    
    if DATASET_TYPE == "petri_dish":
        # 3-class: argmax → body class (1)
        pred = output.argmax(dim=1).squeeze().cpu().numpy()
        pred_full = cv2.resize(pred.astype(np.uint8), (original_shape[1], original_shape[0]),
                               interpolation=cv2.INTER_NEAREST)
        body_mask = (pred_full == 1).astype(np.uint8)
    else:
        # 1-class distance map: threshold to get binary mask
        pred = output.squeeze().cpu().numpy()
        pred_full = cv2.resize(pred, (original_shape[1], original_shape[0]),
                               interpolation=cv2.INTER_LINEAR)
        body_mask = (pred_full > THRESHOLD).astype(np.uint8)
    
    # Connected components → instances
    instance_mask = measure.label(body_mask)
    
    # Remove small objects
    if min_area > 0:
        for region in regionprops(instance_mask):
            if region.area < min_area:
                instance_mask[instance_mask == region.label] = 0
        instance_mask = measure.label(instance_mask > 0)
    
    num_instances = instance_mask.max()
    return instance_mask, num_instances


print(f"\nsegment_boundary_model() defined — ready for {DATASET_TYPE} pipeline.")

## 1.5 Preprocessing Transforms
Define image preprocessing pipelines for model inference.

In [ ]:
# Preprocessing pipeline for inference
# - Resize to 256x256 (model input size)
# - Normalize to [-1, 1] range
# - Convert to PyTorch tensor

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
])

print("Preprocessing transform ready")
print("  - Input: RGB image (any size)")
print("  - Output: Normalized tensor (256x256)")

## 1.6 Load Data
Get image files or set up video input based on `DATASET_TYPE`.

In [ ]:
# Get image files or set up video - based on USE_VIDEO flag
if USE_VIDEO:
    # ===== VIDEO INPUT MODE =====
    image_files = None  # Not used for video
    frames_to_process = total_frames if FRAME_LIMIT is None else min(FRAME_LIMIT, total_frames)
    total_minutes = frames_to_process * MINUTES_PER_FRAME
    total_hours = total_minutes / 60
    
    print(f"VIDEO INPUT MODE")
    print(f"Video: {Path(VIDEO_PATH).name}")
    print(f"Total frames: {total_frames}")
    print(f"Frames to process: {frames_to_process}")
    print(f"Time span: {total_hours:.1f} hours ({total_minutes} minutes)")

else:
    # ===== IMAGE INPUT MODE =====
    image_dir = Path(IMAGE_DIRECTORY)
    all_images = sorted(list(image_dir.glob('*.jpeg')) + list(image_dir.glob('*.png')))
    
    # Select subset if FRAME_LIMIT is specified
    if FRAME_LIMIT is None:
        image_files = all_images
    else:
        image_files = all_images[:FRAME_LIMIT]
    
    # Calculate time span
    total_minutes = len(image_files) * MINUTES_PER_FRAME
    total_hours = total_minutes / 60
    
    print(f"IMAGE INPUT MODE")
    print(f"Total images available: {len(all_images)}")
    print(f"Images to process: {len(image_files)}")
    print(f"Time span: {total_hours:.1f} hours ({total_minutes} minutes)")
    print(f"\nFirst image: {image_files[0].name}")
    print(f"Last image: {image_files[-1].name}")

# Helper function to get reference frame (works for both video and images)
def get_reference_frame(frame_num=0):
    """Get a frame from video or image files for visualization"""
    if USE_VIDEO:
        cap = cv2.VideoCapture(VIDEO_PATH)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame_bgr = cap.read()
        cap.release()
        return frame_bgr
    else:
        return cv2.imread(str(image_files[frame_num]))

---
# PART 2: SEGMENTATION DEMO
---
Test the segmentation on a single image before running the full pipeline.

## 2.1 Single Image Segmentation
Segment one frame and visualize the result.

In [ ]:
# ============================================
# DELIVERABLE 1: Single Image Segmentation
# ============================================
# Shows: Raw image | Segmentation overlay
# Purpose: Prove the model can identify individuals
# Supports both petri dish (images) and microfluidics (video)

from torchvision import transforms as tv_transforms

# Simple preprocessing (for microfluidics / when USE_WATERSHED=False)
raw_transform = tv_transforms.Compose([
    tv_transforms.ToTensor(),
    tv_transforms.Resize((256, 256)),
])

def segment_single_image(image_input, model, device, threshold=0.2, min_distance=7, min_area=30, use_watershed=True):
    """Segment a single image and return results
    
    Args:
        image_input: Either a Path/str to image file, OR an RGB numpy array (for video frames)
        use_watershed: If True, use watershed; if False, use simple threshold
    """
    # Load image if path provided, otherwise use array directly
    if isinstance(image_input, (str, Path)):
        frame_bgr = cv2.imread(str(image_input))
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    else:
        frame_rgb = image_input  # Already RGB numpy array
    
    original_shape = frame_rgb.shape[:2]
    
    if not use_watershed:
        # ===== MULTI-CLASS APPROACH (4-channel model) =====
        input_tensor = raw_transform(frame_rgb).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(input_tensor)
            # 4-channel output: take argmax to get class predictions
            pred_classes = torch.argmax(output, dim=1).squeeze().cpu().numpy()
        
        # Any non-zero class = foreground (class 0 = background)
        binary_mask = (pred_classes > 0).astype(np.uint8)
        binary_mask_full = resize(binary_mask, original_shape, preserve_range=True, order=0).astype(np.uint8)
        
        # Connected components
        instance_mask = measure.label(binary_mask_full)
        num_instances = instance_mask.max()
    else:
        # ===== WATERSHED APPROACH (for petri dish) =====
        augmented = val_transform(image=frame_rgb)
        input_tensor = augmented['image'].unsqueeze(0).to(device)
        
        with torch.no_grad():
            distance_map_pred = model(input_tensor).squeeze().cpu().numpy()
        
        distance_map_full = resize(distance_map_pred, original_shape, preserve_range=True)
        
        instance_mask, num_instances, centroids = distance_map_to_instances(
            distance_map_full,
            threshold=threshold,
            min_distance=min_distance,
            min_area=min_area
        )
    
    return frame_rgb, instance_mask, num_instances

# Pick a sample frame (middle of time-lapse for good representation)
if USE_VIDEO:
    # Video input - grab middle frame
    sample_idx = total_frames // 2
    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, sample_idx)
    ret, frame_bgr = cap.read()
    cap.release()
    sample_frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    print(f"Sample frame: {sample_idx} of {total_frames} (from video)")
else:
    # Image input - grab middle image
    sample_idx = len(image_files) // 2
    sample_image = image_files[sample_idx]
    sample_frame_rgb = None  # Will be loaded by function
    print(f"Sample image: {sample_image.name}")
    print(f"Frame {sample_idx} of {len(image_files)}")

# Run segmentation (use boundary model if distance model was not loaded)
try:
    _ = model
    use_boundary = False
except NameError:
    use_boundary = 'boundary_model' in dir()
    if not use_boundary:
        raise NameError("Run the 'Load Model' (1.3) or 'Load Boundary Model' (1.3b) cell first.")

if use_boundary:
    if USE_VIDEO:
        frame_rgb = sample_frame_rgb.copy()
    else:
        frame_rgb = cv2.cvtColor(cv2.imread(str(sample_image)), cv2.COLOR_BGR2RGB)
    instance_mask, num_instances = segment_boundary_model(frame_rgb, min_area=MIN_AREA)
    print(f"Segmentation method: Boundary model (3-class)")
else:
    print(f"Segmentation method: {'Watershed' if USE_WATERSHED else 'Simple threshold'}")
    if USE_VIDEO:
        frame_rgb, instance_mask, num_instances = segment_single_image(
            sample_frame_rgb, model, device, THRESHOLD, MIN_DISTANCE, MIN_AREA, use_watershed=USE_WATERSHED
        )
    else:
        frame_rgb, instance_mask, num_instances = segment_single_image(
            sample_image, model, device, THRESHOLD, MIN_DISTANCE, MIN_AREA, use_watershed=USE_WATERSHED
        )

print(f"Detected {num_instances} individuals")

In [ ]:
# ============================================
# 2.1b: Single Image Segmentation using BOUNDARY MODEL (NEW)
# ============================================

# Pick sample frame (middle of time-lapse)
if USE_VIDEO:
    sample_idx = total_frames // 2
    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, sample_idx)
    ret, frame_bgr = cap.read()
    cap.release()
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    print(f"Sample frame: {sample_idx} of {total_frames} (from video)")
else:
    sample_idx = len(image_files) // 2
    sample_image = image_files[sample_idx]
    frame_bgr = cv2.imread(str(sample_image))
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    print(f"Sample image: {sample_image.name}")
    print(f"Frame {sample_idx} of {len(image_files)}")

# Run segmentation
instance_mask, num_instances = segment_boundary_model(frame_rgb, min_area=MIN_AREA)
print(f"Detected {num_instances} individuals")

# --- Build bright color map ---
np.random.seed(42)
uid_list = [u for u in np.unique(instance_mask) if u > 0]
colors = {}
for uid in uid_list:
    # Generate saturated, bright colors (HSV with high S and V)
    hue = (uid * 37) % 180  # spread hues
    hsv = np.uint8([[[hue, 220, 230]]])
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)[0][0].tolist()
    colors[uid] = rgb

# --- Build overlay image ---
overlay = frame_rgb.copy().astype(np.float32)
for uid in uid_list:
    mask = (instance_mask == uid)
    c = np.array(colors[uid], dtype=np.float32)
    overlay[mask] = 0.4 * overlay[mask] + 0.6 * c  # strong color fill

overlay = np.clip(overlay, 0, 255).astype(np.uint8)

# Draw thick contours
for uid in uid_list:
    mask = (instance_mask == uid)
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = colors[uid]
    cv2.drawContours(overlay, contours, -1, c, 2)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Left: raw image
axes[0].imshow(frame_rgb)
axes[0].set_title("Raw Image", fontsize=14)
axes[0].axis('off')

# Right: overlay with labels
axes[1].imshow(overlay)
for uid in uid_list:
    mask = (instance_mask == uid)
    coords = np.where(mask)
    cy, cx = coords[0].mean(), coords[1].mean()  # centroid
    c = [v/255 for v in colors[uid]]
    axes[1].text(cx, cy, str(uid), fontsize=7, color='white', fontweight='bold',
                 ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.2', fc=c, ec='white', lw=0.5, alpha=0.85))

axes[1].set_title(f"Segmentation: {num_instances} fronds detected", fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

---
# PART 3: FULL PIPELINE
---
Process all frames, segment individuals, and track over time.

## 3.1b Segmentation & Tracking with Boundary Model (NEW)
Use the 3-class boundary U-Net (`segment_boundary_model`) for the full pipeline.
This overrides `all_detections` and `tracks` produced by the original 3.1 cell.

In [ ]:
# ============================================
# FULL PIPELINE WITH BOUNDARY MODEL
# ============================================
# This re-runs segmentation + tracking using segment_boundary_model()
# and overwrites all_detections and tracks.

import logging
import io
import contextlib
from tqdm.auto import tqdm

logging.getLogger('btrack').setLevel(logging.ERROR)

print("=" * 60)
print("RUNNING SEGMENTATION & TRACKING PIPELINE (BOUNDARY MODEL)")
print("=" * 60)
print(f"Mode: {'VIDEO' if USE_VIDEO else 'IMAGES'}")
print("Segmentation: Boundary model (3-class)")

# Setup btrack tracker
if Path(BTRACK_CONFIG).exists():
    tracker = btrack.BayesianTracker()
    tracker.configure_from_file(BTRACK_CONFIG)
    print(f"Config: {BTRACK_CONFIG}")
else:
    tracker = btrack.BayesianTracker()
    print("Config: default")
tracker.max_search_radius = 50

all_detections = []

# Frames to process
if USE_VIDEO:
    total_frames_to_process = total_frames if FRAME_LIMIT is None else min(FRAME_LIMIT, total_frames)
    cap = cv2.VideoCapture(VIDEO_PATH)
else:
    total_frames_to_process = len(image_files)

pbar = tqdm(total=total_frames_to_process, desc="Boundary seg+tracking", unit="frame")

frame_idx = 0
while True:
    if USE_VIDEO:
        ret, frame_bgr = cap.read()
        if not ret or (FRAME_LIMIT and frame_idx >= FRAME_LIMIT):
            break
        frame_name = f"frame_{frame_idx:04d}"
    else:
        if frame_idx >= len(image_files):
            break
        img_path = image_files[frame_idx]
        frame_bgr = cv2.imread(str(img_path))
        frame_name = img_path.name

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Use boundary model for segmentation
    instance_mask, num_instances = segment_boundary_model(frame_rgb, min_area=MIN_AREA)

    # Region properties
    props = regionprops(instance_mask)

    for prop in props:
        all_detections.append({
            'frame': frame_idx,
            'image_file': frame_name,
            'label': prop.label,
            'area': prop.area,
            'perimeter': prop.perimeter,
            'major_axis_length': prop.major_axis_length,
            'centroid_y': prop.centroid[0],
            'centroid_x': prop.centroid[1],
            'bbox_minr': prop.bbox[0],
            'bbox_minc': prop.bbox[1],
            'bbox_maxr': prop.bbox[2],
            'bbox_maxc': prop.bbox[3],
        })

    # Send to btrack
    labeled_mask_3d = np.zeros((1, *instance_mask.shape), dtype=instance_mask.dtype)
    labeled_mask_3d[0] = instance_mask
    with contextlib.redirect_stderr(io.StringIO()):
        objects = segmentation_to_objects(labeled_mask_3d, properties=("centroid",))
    for o in objects:
        o.t = frame_idx
    tracker.append(objects)

    frame_idx += 1
    pbar.update(1)

pbar.close()

if USE_VIDEO:
    cap.release()

print(f"\nSegmentation complete (boundary model): {len(all_detections)} detections across {frame_idx} frames")

print("Running btrack tracking (boundary model)...")
tracker.track_interactive(step_size=50)
tracker.optimize()
tracks = tracker.tracks
print(f"Tracking complete: {len(tracks)} tracks found (boundary model)")

## 3.1c Save Segmentation Video (Boundary Model)
Re-runs segmentation frame-by-frame and writes an MP4 with colored instance masks, bounding boxes, numbered frond labels, and a frame/time/frond-count HUD.

In [ ]:
# ============================================
# 3.1c: SAVE SEGMENTATION OVERLAY VIDEO
# ============================================
# Produces an MP4 matching the reference style:
#   - Grayscale background
#   - Semi-transparent colored instance masks
#   - Yellow numbered ID labels on each frond
#   - Black HUD box (top-left): Frame, Time, Fronds count (green)

from tqdm.auto import tqdm

SEG_VIDEO_PATH = OUTPUT_DIR + "segmentation_boundary_overlay.mp4"
SEG_VIDEO_FPS = 10  # output playback speed

# --- helpers -----------------------------------------------------------------

def _make_frond_colors(n, seed=42):
    """Generate n distinct, saturated colors via evenly-spaced HSV hues."""
    np.random.seed(seed)
    colors = {}
    for i in range(1, n + 1):
        hue = int((i * 37) % 180)
        hsv = np.uint8([[[hue, 220, 230]]])
        rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)[0][0]
        colors[i] = tuple(int(c) for c in rgb)
    return colors

def _draw_overlay(frame_rgb, instance_mask, frame_idx, minutes_per_frame, alpha=0.55):
    """Draw colored masks, labels, and HUD onto a grayscale copy."""
    # Grayscale background (3-channel so we can draw color on top)
    gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
    canvas = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

    uid_list = sorted([u for u in np.unique(instance_mask) if u > 0])
    num_fronds = len(uid_list)
    colors = _make_frond_colors(max(uid_list) if uid_list else 0)

    for uid in uid_list:
        mask = (instance_mask == uid)
        c = colors.get(uid, (200, 200, 200))

        # Semi-transparent color fill
        roi = canvas[mask].astype(np.float32)
        canvas[mask] = np.clip((1 - alpha) * roi + alpha * np.array(c, dtype=np.float32), 0, 255).astype(np.uint8)

        # Numbered label (yellow text with small dark shadow)
        ys, xs = np.where(mask)
        cx, cy = int(xs.mean()), int(ys.mean())
        label = str(uid)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.7
        thickness = 2
        # shadow
        cv2.putText(canvas, label, (cx - 1, cy + 1), font, font_scale, (0, 0, 0), thickness + 1, cv2.LINE_AA)
        # foreground
        cv2.putText(canvas, label, (cx, cy), font, font_scale, (0, 255, 255), thickness, cv2.LINE_AA)

    # --- HUD (top-left black box) ---
    total_minutes = frame_idx * minutes_per_frame
    hours = total_minutes // 60
    mins = total_minutes % 60
    time_str = f"{int(hours):02d}:{int(mins):02d}" if hours > 0 else f"{int(mins):02d}:00"

    lines = [
        f"Frame: {frame_idx}",
        f"Time: {time_str} (min {int(total_minutes)})",
    ]
    frond_line = f"Fronds: {num_fronds}"

    font = cv2.FONT_HERSHEY_SIMPLEX
    fs, th = 0.7, 2
    line_h = 30
    box_w = 340
    box_h = line_h * (len(lines) + 1) + 12
    cv2.rectangle(canvas, (0, 0), (box_w, box_h), (0, 0, 0), -1)

    for i, line in enumerate(lines):
        cv2.putText(canvas, line, (8, 24 + i * line_h), font, fs, (255, 255, 255), th, cv2.LINE_AA)
    # Fronds line in green
    cv2.putText(canvas, frond_line, (8, 24 + len(lines) * line_h), font, fs, (0, 255, 0), th, cv2.LINE_AA)

    return canvas

# --- main loop ---------------------------------------------------------------

print("=" * 60)
print("SAVING SEGMENTATION OVERLAY VIDEO (BOUNDARY MODEL)")
print("=" * 60)

# Determine frame source
if USE_VIDEO:
    cap = cv2.VideoCapture(VIDEO_PATH)
    n_frames = total_frames if FRAME_LIMIT is None else min(FRAME_LIMIT, total_frames)
else:
    n_frames = len(image_files)

# Read first frame to get dimensions
if USE_VIDEO:
    ret, probe = cap.read()
    h, w = probe.shape[:2]
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
else:
    probe = cv2.imread(str(image_files[0]))
    h, w = probe.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(SEG_VIDEO_PATH, fourcc, SEG_VIDEO_FPS, (w, h))

pbar = tqdm(total=n_frames, desc="Writing seg video", unit="frame")

for fidx in range(n_frames):
    # Load frame
    if USE_VIDEO:
        ret, frame_bgr = cap.read()
        if not ret:
            break
    else:
        frame_bgr = cv2.imread(str(image_files[fidx]))

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Segment
    instance_mask, _ = segment_boundary_model(frame_rgb, min_area=MIN_AREA)

    # Draw overlay (returns RGB)
    overlay_rgb = _draw_overlay(frame_rgb, instance_mask, fidx, MINUTES_PER_FRAME)

    # Write (BGR)
    writer.write(cv2.cvtColor(overlay_rgb, cv2.COLOR_RGB2BGR))
    pbar.update(1)

pbar.close()
writer.release()
if USE_VIDEO:
    cap.release()

print(f"\nVideo saved: {SEG_VIDEO_PATH}")
print(f"Frames: {n_frames} | FPS: {SEG_VIDEO_FPS} | Duration: {n_frames / SEG_VIDEO_FPS:.1f}s")

## 3.2 Save Checkpoint
**Run after tracking completes!** Saves data to avoid re-running.

In [ ]:
# ============================================
# SAVE TRACKING DATA - Run after tracking!
# ============================================
import pickle

checkpoint_path = OUTPUT_DIR + 'tracking_checkpoint.pkl'

# Check if data exists
has_unified = 'unified_df' in dir() and unified_df is not None and len(unified_df) > 0
has_lineage = 'lineage_df' in dir() and lineage_df is not None
has_extracted = 'extracted_df' in dir() and extracted_df is not None and len(extracted_df) > 0

if not has_unified and not has_extracted:
    print('⚠️  WARNING: No tracking data found!')
    print('   Run the tracking cell above first.')
else:
    checkpoint_data = {
        'unified_df': unified_df if has_unified else None,
        'lineage_df': lineage_df if has_lineage else None,
        'extracted_df': extracted_df if has_extracted else None,
        'population_per_frame_filtered': population_per_frame_filtered if 'population_per_frame_filtered' in dir() else None,
        'extracted_df_filtered': extracted_df_filtered if 'extracted_df_filtered' in dir() else None,
        'DISH_ROI': DISH_ROI if 'DISH_ROI' in dir() else None,
        'MINUTES_PER_FRAME': MINUTES_PER_FRAME if 'MINUTES_PER_FRAME' in dir() else 5,
        'IMAGE_DIRECTORY': IMAGE_DIRECTORY if 'IMAGE_DIRECTORY' in dir() else None,
        'OUTPUT_DIR': OUTPUT_DIR if 'OUTPUT_DIR' in dir() else None,
    }
    
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint_data, f)
    
    print('✓ TRACKING DATA SAVED!')
    print(f'  Path: {checkpoint_path}')
    print(f'  • unified_df: {len(unified_df)} rows' if has_unified else '  • unified_df: None')
    print(f'  • lineage_df: {len(lineage_df)} rows' if has_lineage else '  • lineage_df: None')
    print(f'  • extracted_df: {len(extracted_df)} rows' if has_extracted else '  • extracted_df: None')
    print('\n📌 After kernel restart, run cells 1-5 then the LOAD cell.')


## 3.3 Create Unified DataFrame
Combine segmentation data with tracking IDs.

In [ ]:
# ============================================
# CREATE UNIFIED TRACKING DATAFRAME
# ============================================

# Convert tracks to DataFrame
track_data = []
for tr in tracks:
    if len(tr) < 2:  # Skip very short tracks
        continue
    for x, y, t in zip(tr['x'], tr['y'], tr['t']):
        track_data.append({
            'track_id': tr.ID,
            'track_x': x,
            'track_y': y,
            'frame': t
        })
track_df = pd.DataFrame(track_data)

# Detection DataFrame
detection_df = pd.DataFrame(all_detections)

# Match detections to tracks (by proximity)
unified_data = []
for _, det in detection_df.iterrows():
    frame = det['frame']
    tracks_in_frame = track_df[track_df['frame'] == frame]
    
    matched_track_id = None
    if not tracks_in_frame.empty:
        # Find closest track
        distances = np.sqrt(
            (tracks_in_frame['track_x'] - det['centroid_x'])**2 + 
            (tracks_in_frame['track_y'] - det['centroid_y'])**2
        )
        if len(distances) > 0:
            min_idx = distances.idxmin()
            if distances[min_idx] < 50:  # Max matching distance
                matched_track_id = tracks_in_frame.loc[min_idx, 'track_id']
    
    unified_data.append({
        'frame': int(det['frame']),
        'image_file': det['image_file'],
        'track_id': matched_track_id,
        'detection_label': int(det['label']),
        # Measurements in mm (conversion factors from calibration)
        'area_mm2': det['area'] * 0.00111,
        'perimeter_mm': det['perimeter'] * 0.03333,
        'major_axis_mm': det['major_axis_length'] * 0.03333,
        # Measurements in pixels
        'area_pixels': det['area'],
        'centroid_x': det['centroid_x'],
        'centroid_y': det['centroid_y'],
        # Bounding box
        'bbox_minr': det['bbox_minr'],
        'bbox_minc': det['bbox_minc'],
        'bbox_maxr': det['bbox_maxr'],
        'bbox_maxc': det['bbox_maxc'],
    })

# Create DataFrame
unified_df = pd.DataFrame(unified_data)
unified_df = unified_df.sort_values(['track_id', 'frame']).reset_index(drop=True)

# Add time columns
unified_df['minutes'] = unified_df['frame'] * MINUTES_PER_FRAME
unified_df['hours'] = unified_df['minutes'] / 60

print(f"Unified DataFrame created:")
print(f"  Total detections: {len(unified_df)}")
print(f"  Unique tracks: {unified_df['track_id'].nunique()}")
print(f"  Frames: {unified_df['frame'].nunique()}")
print(f"\nColumns: {list(unified_df.columns)}")
unified_df.head()

## 3.4 Infer Lineage
Identify parent-child relationships (budding events).

In [ ]:
# ============================================
# TRACK STITCHING + LINEAGE INFERENCE
# ============================================
# Step 1: Stitch fragmented tracks (same plant, different IDs)
# Step 2: Detect TRUE budding events (population increase)

def stitch_fragmented_tracks(track_df, unified_df, max_gap_frames=15, max_stitch_dist=20):
    """
    Identify and merge fragmented tracks (same plant, different track IDs).
    
    A track is considered a continuation if:
    - It starts near where another track ended (within max_stitch_dist pixels)
    - The gap is within max_gap_frames
    - Population did NOT increase during this transition (1→1, not 1→2)
    
    Returns a mapping: {new_track_id: original_track_id}
    """
    print("="*50)
    print("TRACK STITCHING (Merging Fragmented Tracks)")
    print("="*50)
    
    # Get track info
    track_groups = track_df.groupby('track_id')
    track_info = track_groups.agg({
        'frame': ['min', 'max'],
        'track_x': ['first', 'last'],
        'track_y': ['first', 'last']
    }).reset_index()
    track_info.columns = ['track_id', 'start_frame', 'end_frame', 'start_x', 'end_x', 'start_y', 'end_y']
    
    # Population per frame (to check if it's continuation vs budding)
    pop_per_frame = unified_df.groupby('frame')['track_id'].nunique()
    
    # Build a mapping: track_id -> canonical_id (the first track in a chain)
    track_to_canonical = {tid: tid for tid in track_info['track_id']}
    stitch_chains = []  # For logging
    
    # Sort by start frame
    track_info_sorted = track_info.sort_values('start_frame')
    
    for _, new_track in track_info_sorted.iterrows():
        new_id = new_track['track_id']
        new_start = int(new_track['start_frame'])
        new_pos = np.array([new_track['start_x'], new_track['start_y']])
        
        # Look for tracks that ended before this one started
        potential_predecessors = track_info[
            (track_info['end_frame'] < new_start) & 
            (track_info['end_frame'] >= new_start - max_gap_frames) &
            (track_info['track_id'] != new_id)
        ]
        
        best_match = None
        best_dist = float('inf')
        
        for _, pred in potential_predecessors.iterrows():
            pred_id = pred['track_id']
            pred_end = int(pred['end_frame'])
            pred_pos = np.array([pred['end_x'], pred['end_y']])
            
            dist = np.linalg.norm(new_pos - pred_pos)
            
            if dist < max_stitch_dist and dist < best_dist:
                # For very close tracks (same position), always allow stitching
                # For farther tracks, check local population
                if dist < 5:  # Essentially same position - definitely a continuation
                    best_dist = dist
                    best_match = pred_id
                else:
                    # Check LOCAL population near this track (not global)
                    pred_pos = np.array([pred['end_x'], pred['end_y']])
                    local_radius = 50
                    
                    # Count tracks near this position at pred_end vs new_start
                    local_before = unified_df[
                        (unified_df['frame'] == pred_end) &
                        (np.sqrt((unified_df['centroid_x'] - pred_pos[0])**2 + 
                                 (unified_df['centroid_y'] - pred_pos[1])**2) < local_radius)
                    ]['track_id'].nunique()
                    
                    local_after = unified_df[
                        (unified_df['frame'] == new_start) &
                        (np.sqrt((unified_df['centroid_x'] - new_pos[0])**2 + 
                                 (unified_df['centroid_y'] - new_pos[1])**2) < local_radius)
                    ]['track_id'].nunique()
                    
                    # Allow if local population didn't increase much
                    if local_after <= local_before + 2:
                        best_dist = dist
                        best_match = pred_id
        
        if best_match is not None:
            # Stitch: new_id continues from best_match
            # Find the canonical ID of best_match
            canonical = track_to_canonical[best_match]
            track_to_canonical[new_id] = canonical
            stitch_chains.append((canonical, best_match, new_id, best_dist))
    
    # Count stitches
    unique_canonicals = set(track_to_canonical.values())
    n_stitched = len(track_to_canonical) - len(unique_canonicals)
    
    print(f"Tracks analyzed: {len(track_info)}")
    print(f"Tracks stitched: {n_stitched}")
    print(f"Unique plants after stitching: {len(unique_canonicals)}")
    
    if stitch_chains:
        print(f"\nStitch chains (first 10):")
        for chain in stitch_chains[:10]:
            canonical, pred, new, dist = chain
            print(f"  Track {int(new)} -> continues Track {int(pred)} (dist={dist:.1f}px, canonical={int(canonical)})")
    
    return track_to_canonical

def infer_lineage_improved(track_df, unified_df, max_frame_gap=5, max_dist=50, track_mapping=None):
    """
    Improved lineage inference that only detects real budding events.
    
    Key improvements:
    1. Only counts as budding when population INCREASED at that frame
    2. Distinguishes budding (1→2+) from track continuity (1→1)
    3. Handles budding: mother continues + daughter appears
    
    Parameters:
    -----------
    track_df : DataFrame with track_id, frame, track_x, track_y
    unified_df : Full tracking data to count population per frame
    max_frame_gap : Max frames between parent end and child start
    max_dist : Max distance (px) between parent's last position and child's first
    """
    
    # Step 1: Calculate population per frame
    pop_per_frame = unified_df.groupby('frame')['track_id'].nunique()
    
    # Step 2: Find frames where population increased (potential budding frames)
    pop_diff = pop_per_frame.diff().fillna(0)
    division_frames = set(pop_diff[pop_diff > 0].index.tolist())
    print(f"Frames with population increase: {len(division_frames)}")
    
    # Step 3: Get track info
    track_groups = track_df.groupby('track_id')
    track_info = track_groups.agg({
        'frame': ['min', 'max'],
        'track_x': ['first', 'last'],
        'track_y': ['first', 'last']
    }).reset_index()
    track_info.columns = ['track_id', 'start_frame', 'end_frame', 'start_x', 'end_x', 'start_y', 'end_y']
    
    # Also get average area for size matching
    if 'area_pixels' in unified_df.columns:
        track_areas = unified_df.groupby('track_id')['area_pixels'].mean().to_dict()
    else:
        track_areas = {}
    
    # Step 4: Find new tracks that appeared at budding frames
    lineage_pairs = []
    
    for div_frame in sorted(division_frames):
        # New tracks that started at or just after this frame
        new_tracks = track_info[
            (track_info['start_frame'] >= div_frame) & 
            (track_info['start_frame'] <= div_frame + max_frame_gap)
        ]
        
        if len(new_tracks) == 0:
            continue
        
        # Potential parents: tracks that existed just before this frame
        # Either ended near this frame, OR were still active (budding case)
        potential_parents = track_info[
            # Parent ended just before new track appeared
            ((track_info['end_frame'] >= div_frame - max_frame_gap) & 
             (track_info['end_frame'] < div_frame + max_frame_gap)) |
            # OR parent was still active (budding: mother continues)
            ((track_info['start_frame'] < div_frame) & 
             (track_info['end_frame'] >= div_frame))
        ]
        
        for _, new_track in new_tracks.iterrows():
            new_id = new_track['track_id']
            new_start = int(new_track['start_frame'])
            new_pos = np.array([new_track['start_x'], new_track['start_y']])
            
            best_parent = None
            best_dist = float('inf')
            
            for _, parent in potential_parents.iterrows():
                parent_id = parent['track_id']
                if parent_id == new_id:
                    continue
                
                # Get parent position at the relevant time
                parent_end = int(parent['end_frame'])
                
                # For still-active parents (budding), get position near new track's start
                if parent_end >= new_start:
                    # Parent was active when child appeared - get parent pos at that frame
                    parent_at_frame = track_df[
                        (track_df['track_id'] == parent_id) & 
                        (track_df['frame'] <= new_start)
                    ]
                    if len(parent_at_frame) > 0:
                        last_row = parent_at_frame.iloc[-1]
                        parent_pos = np.array([last_row['track_x'], last_row['track_y']])
                    else:
                        continue
                else:
                    # Parent ended before child started
                    parent_pos = np.array([parent['end_x'], parent['end_y']])
                
                dist = np.linalg.norm(new_pos - parent_pos)
                
                if dist < max_dist and dist < best_dist:
                    best_dist = dist
                    best_parent = parent_id
            
            if best_parent is not None:
                lineage_pairs.append({
                    'parent_id': best_parent,
                    'child_id': new_id,
                    'division_frame': new_start,
                    'distance': best_dist
                })
    
    lineage_df = pd.DataFrame(lineage_pairs).drop_duplicates(subset=['child_id'])
    
    # Step 5: Filter - only keep if parent has 2+ children OR parent continues after child
    if len(lineage_df) > 0:
        parent_child_counts = lineage_df.groupby('parent_id').size()
        
        valid_pairs = []
        for _, row in lineage_df.iterrows():
            parent_id = row['parent_id']
            child_id = row['child_id']
            div_frame = row['division_frame']
            
            # Check if parent has multiple children
            has_multiple_children = parent_child_counts.get(parent_id, 0) >= 2
            
            # Check if parent continues after child appeared (budding)
            parent_info = track_info[track_info['track_id'] == parent_id]
            if len(parent_info) > 0:
                parent_continues = parent_info.iloc[0]['end_frame'] > div_frame
            else:
                parent_continues = False
            
            # Keep if either condition is met
            if has_multiple_children or parent_continues:
                valid_pairs.append(row)
        
        lineage_df = pd.DataFrame(valid_pairs)
    
    print(f"Valid budding events (filtered): {len(lineage_df)}")
    
    return lineage_df

# ============================================
# STEP 1: STITCH FRAGMENTED TRACKS
# ============================================
track_mapping = stitch_fragmented_tracks(track_df, unified_df, max_gap_frames=15, max_stitch_dist=20)

# Create a "canonical" track ID column (maps fragmented tracks to their original)
unified_df['canonical_track_id'] = unified_df['track_id'].map(track_mapping)
track_df_copy = track_df.copy()
track_df_copy['canonical_track_id'] = track_df_copy['track_id'].map(track_mapping)

# ============================================
# STEP 2: LINEAGE INFERENCE (on canonical tracks)
# ============================================
print("\n")
print("="*50)
print("IMPROVED LINEAGE INFERENCE")
print("="*50)

# For lineage, we work with canonical IDs to connect stitched tracks
lineage_df = infer_lineage_improved(track_df, unified_df, max_frame_gap=5, max_dist=50)

# Remap lineage to canonical IDs
if len(lineage_df) > 0:
    lineage_df['parent_canonical'] = lineage_df['parent_id'].map(track_mapping)
    lineage_df['child_canonical'] = lineage_df['child_id'].map(track_mapping)
    
    # Remove self-links (parent and child are same canonical track = stitched continuation)
    lineage_df_clean = lineage_df[lineage_df['parent_canonical'] != lineage_df['child_canonical']].copy()
    
    # Deduplicate by canonical IDs
    lineage_df_clean = lineage_df_clean.drop_duplicates(subset=['parent_canonical', 'child_canonical'])
    
    print(f"\nAfter removing stitched continuations:")
    print(f"  Original links: {len(lineage_df)}")
    print(f"  True budding events: {len(lineage_df_clean)}")
    
    lineage_df = lineage_df_clean
else:
    lineage_df['parent_canonical'] = []
    lineage_df['child_canonical'] = []

# Add lineage info to unified_df (using canonical IDs)
parent_map_raw = dict(zip(lineage_df['child_canonical'], lineage_df['parent_canonical']))

# Remove any cycles from parent_map
def remove_cycles(parent_map):
    """Remove cycles from parent-child mapping"""
    clean_map = {}
    removed = []
    
    for child, parent in parent_map.items():
        # Check if adding this creates a cycle
        seen = {child}
        current = parent
        has_cycle = False
        
        while current in parent_map:
            if current in seen or current == child:
                has_cycle = True
                break
            seen.add(current)
            current = parent_map.get(current)
        
        if has_cycle:
            removed.append((child, parent))
        else:
            clean_map[child] = parent
    
    if removed:
        print(f"Removed {len(removed)} cycle-creating links:")
        for child, parent in removed[:5]:
            print(f"  {child} -> {parent}")
    
    return clean_map

parent_map = remove_cycles(parent_map_raw)
unified_df['parent_track_id'] = unified_df['canonical_track_id'].map(parent_map)

# Calculate generation (based on canonical track IDs) - with cycle detection
def get_generation(track_id, parent_map, memo=None, visited=None):
    if memo is None:
        memo = {}
    if visited is None:
        visited = set()
    
    if pd.isna(track_id):
        return None
    if track_id in memo:
        return memo[track_id]
    if track_id not in parent_map:
        memo[track_id] = 0
        return 0
    
    # Cycle detection
    if track_id in visited:
        print(f"  Warning: Cycle detected at track {track_id}, breaking chain")
        memo[track_id] = 0
        return 0
    
    visited.add(track_id)
    parent = parent_map[track_id]
    generation = 1 + get_generation(parent, parent_map, memo, visited.copy())
    memo[track_id] = generation
    return generation

gen_memo = {}
unified_df['generation'] = unified_df['canonical_track_id'].apply(
    lambda x: get_generation(x, parent_map, gen_memo) if pd.notna(x) else None
)

# Check for any cycles in parent_map
print("\nChecking for cycles in lineage...")
for child, parent in parent_map.items():
    if child == parent:
        print(f"  Self-reference: {child} -> {parent}")
    # Check for longer cycles
    seen = {child}
    current = parent
    while current in parent_map and current not in seen:
        seen.add(current)
        current = parent_map[current]
    if current in seen:
        print(f"  Cycle found: {child} -> ... -> {current}")

print(f"\nLineage inference complete:")
print(f"  Budding events found: {len(lineage_df)}")
print(f"  Max generation: {unified_df['generation'].max()}")

if len(lineage_df) > 0:
    print(f"\nLineage relationships (canonical IDs):")
    display(lineage_df[['parent_canonical', 'child_canonical', 'division_frame', 'distance']])
else:
    print("\nNo valid budding events found.")

## 3.5 Save Tracking Data
Export processed data as CSV.

In [ ]:
# ============================================
# SAVE TRACKING DATA
# ============================================

# Save unified tracking data
unified_path = OUTPUT_DIR + "unified_tracking_data.csv"
unified_df.to_csv(unified_path, index=False)
print(f"Saved: {unified_path}")

# Save lineage data
lineage_path = OUTPUT_DIR + "lineage_data.csv"
lineage_df.to_csv(lineage_path, index=False)
print(f"Saved: {lineage_path}")

# Save track data
track_path = OUTPUT_DIR + "track_data.csv"
track_df.to_csv(track_path, index=False)
print(f"Saved: {track_path}")

print("\nAll tracking data saved! You can load these files to skip the processing steps.")

---
# PART 4: CLUSTER ANALYSIS
---
Group tracks into spatial clusters and analyze individual clusters.

## 4.1 Identify Clusters
Group nearby tracks into clusters (wells or family groups).

In [ ]:
# ============================================
# IDENTIFY CLUSTERS (Family Groups)
# ============================================
from scipy.cluster.hierarchy import fclusterdata

# Get mean position for each track
track_positions = unified_df.groupby('track_id').agg({
    'centroid_x': 'mean',
    'centroid_y': 'mean'
}).dropna()

# Cluster tracks by spatial proximity
if len(track_positions) > 1:
    if 'N_CLUSTERS' in dir() and N_CLUSTERS is not None and N_CLUSTERS > 0:
        # Use KMeans with known number of clusters (e.g. 6 petri dishes)
        from sklearn.cluster import KMeans
        print(f"Clustering with KMeans (n_clusters={N_CLUSTERS})")
        kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
        clusters = kmeans.fit_predict(track_positions[['centroid_x', 'centroid_y']].values) + 1
    else:
        # Fallback: distance-based hierarchical clustering
        print(f"Clustering with distance threshold: {CLUSTER_DISTANCE} pixels")
        clusters = fclusterdata(track_positions[['centroid_x', 'centroid_y']].values, 
                                t=CLUSTER_DISTANCE, criterion='distance')
    
    track_positions['cluster'] = clusters
    
    # Map clusters back to unified_df
    cluster_map = dict(zip(track_positions.index, track_positions['cluster']))
    unified_df['cluster'] = unified_df['track_id'].map(cluster_map)
    
    # Print cluster summary
    cluster_sizes = unified_df.groupby('cluster')['track_id'].nunique()
    largest_cluster = cluster_sizes.idxmax()
    
    print(f"Found {len(cluster_sizes)} clusters")
    print(f"Cluster sizes: {dict(cluster_sizes)}")
    
    # Print cluster centers
    cluster_summary = track_positions.groupby('cluster').agg({
        'centroid_x': 'mean', 'centroid_y': 'mean'
    })
    for c_id, row in cluster_summary.iterrows():
        n = cluster_sizes.get(c_id, 0)
        print(f"  Cluster {int(c_id)}: {n} tracks at ({row['centroid_x']:.0f}, {row['centroid_y']:.0f})")
    
    print(f"\nLargest cluster: {largest_cluster} ({cluster_sizes[largest_cluster]} tracks)")
else:
    unified_df['cluster'] = 1
    largest_cluster = 1
    print("Only one cluster identified")


## 4.2 Visualize Clusters
View all clusters and select region of interest (ROI).

In [ ]:
# ============================================
# VISUALIZE ALL CLUSTERS & SELECT DISH REGION
# ============================================

# Load reference image (first frame) - works for both video and images
ref_img_rgb = get_reference_frame(0)

# Use global viz parameters from config
LABEL_FONTSIZE = VIZ_LABEL_FONTSIZE
SCATTER_SIZE = VIZ_SCATTER_SIZE
TITLE_FONTSIZE = VIZ_TITLE_FONTSIZE
AXIS_FONTSIZE = VIZ_AXIS_FONTSIZE
LEGEND_FONTSIZE = VIZ_LEGEND_FONTSIZE
FIG_SIZE = VIZ_FIG_SIZE

# Get cluster centers and sizes
cluster_summary = unified_df.groupby('cluster').agg({
    'centroid_x': 'mean',
    'centroid_y': 'mean',
    'track_id': 'nunique'
}).dropna().reset_index()
cluster_summary.columns = ['cluster', 'center_x', 'center_y', 'num_tracks']

print("="*60)
print(f"CLUSTER SUMMARY ({DATASET_TYPE.upper()})")
print("="*60)
print(f"\nTotal clusters found: {len(cluster_summary)}")
print("\nCluster details:")
for _, row in cluster_summary.iterrows():
    print(f"  Cluster {int(row['cluster'])}: {int(row['num_tracks'])} tracks at ({int(row['center_x'])}, {int(row['center_y'])})")

# Visualize all clusters on reference image
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.imshow(ref_img_rgb)

# Plot each cluster with different color and label
colors = plt.cm.Set3(np.linspace(0, 1, len(cluster_summary)))
for idx, row in cluster_summary.iterrows():
    cluster_id = int(row['cluster'])
    x, y = row['center_x'], row['center_y']
    num_tracks = int(row['num_tracks'])
    
    # Get all detections in this cluster (use first available frame)
    cluster_data = unified_df[unified_df['cluster'] == cluster_id]
    first_frame = cluster_data['frame'].min()
    cluster_data_first = cluster_data[cluster_data['frame'] == first_frame]
    
    ax.scatter(cluster_data_first['centroid_x'], cluster_data_first['centroid_y'], 
              c=[colors[idx]], s=SCATTER_SIZE, alpha=0.7, label=f'Cluster {cluster_id} ({num_tracks} tracks)')
    
    # Label cluster center
    ax.annotate(f'C{cluster_id}', (x, y), xytext=(2, 2), 
               textcoords='offset points', fontsize=LABEL_FONTSIZE, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.2', facecolor=colors[idx], alpha=0.7))

ax.set_title(f'All Clusters ({DATASET_TYPE.replace("_", " ").title()})\n(Click to see coordinates, or specify ROI below)', 
             fontsize=TITLE_FONTSIZE, fontweight='bold')
ax.set_xlabel('X (pixels)', fontsize=AXIS_FONTSIZE)
ax.set_ylabel('Y (pixels)', fontsize=AXIS_FONTSIZE)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=LEGEND_FONTSIZE)
ax.tick_params(axis='both', which='major', labelsize=AXIS_FONTSIZE)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nImage dimensions: {ref_img_rgb.shape[1]} x {ref_img_rgb.shape[0]} pixels")
print("\n" + "="*60)
print("TO SELECT A CLUSTER:")
print("="*60)
print("Set SELECTED_CLUSTER in the next cell to analyze a specific cluster")
print("Or set to None to analyze all clusters")

In [ ]:
# ============================================
# EXTRACT CLUSTER FROM SPECIFIC DISH
# ============================================
# 
# 👇 EDIT THESE TWO LINES BELOW TO SELECT YOUR CLUSTER 👇
#

# STEP 1: Set dish region (ROI) - coordinates from visualization above
#   - Set to None to use full image
#   - Or specify: {'x_min': 100, 'x_max': 800, 'y_min': 200, 'y_max': 1000}
DISH_ROI = None

# STEP 2: Set target cluster ID - pick from available clusters shown above
#   - From cell 28 output: Available clusters are 1, 2, 3, 4, 5, 6, 7
#   - Or set to None to see all clusters in the dish region
TARGET_CLUSTER_ID = 6  # Example: 2 (for cluster 2)

# Function to filter data by dish region and cluster
def extract_cluster_from_dish(unified_df, dish_roi=None, cluster_id=None):
    """
    Extract a specific cluster from a specific dish region.
    
    Parameters:
    -----------
    unified_df : DataFrame
        The unified tracking dataframe
    dish_roi : dict or None
        Dictionary with keys 'x_min', 'x_max', 'y_min', 'y_max' defining dish region
        If None, uses full image
    cluster_id : int or None
        Specific cluster ID to extract. If None, returns all clusters in dish
    
    Returns:
    --------
    extracted_df : DataFrame
        Filtered dataframe containing the specified cluster(s) in the dish region
    """
    filtered_df = unified_df.copy()
    
    # Filter by dish region
    if dish_roi is not None:
        mask = (
            (filtered_df['centroid_x'] >= dish_roi['x_min']) &
            (filtered_df['centroid_x'] <= dish_roi['x_max']) &
            (filtered_df['centroid_y'] >= dish_roi['y_min']) &
            (filtered_df['centroid_y'] <= dish_roi['y_max'])
        )
        filtered_df = filtered_df[mask].copy()
        print(f"Dish ROI applied: x=[{dish_roi['x_min']}, {dish_roi['x_max']}], y=[{dish_roi['y_min']}, {dish_roi['y_max']}]")
        print(f"  Detections in dish: {len(filtered_df)}")
    else:
        print("Using full image (no dish ROI)")
    
    # Filter by cluster ID
    if cluster_id is not None:
        filtered_df = filtered_df[filtered_df['cluster'] == cluster_id].copy()
        print(f"\nCluster {cluster_id} selected")
    else:
        # Show available clusters in this region
        clusters_in_region = filtered_df['cluster'].dropna().unique()
        print(f"\nAvailable clusters in region: {sorted([int(c) for c in clusters_in_region])}")
    
    return filtered_df

# Extract the specified cluster from the dish
extracted_df = extract_cluster_from_dish(unified_df, DISH_ROI, TARGET_CLUSTER_ID)

if len(extracted_df) > 0:
    print(f"\n{'='*60}")
    print(f"EXTRACTED DATA SUMMARY")
    print(f"{'='*60}")
    print(f"Total detections: {len(extracted_df)}")
    print(f"Unique tracks: {extracted_df['track_id'].nunique()}")
    print(f"Frames covered: {extracted_df['frame'].nunique()}")
    print(f"Cluster ID(s): {sorted(extracted_df['cluster'].dropna().unique())}")
    print(f"\nTime span: {extracted_df['hours'].min():.1f}h - {extracted_df['hours'].max():.1f}h")
    print(f"{'='*60}\n")
    
    # Preview extracted data
    print("Preview of extracted data:")
    display(extracted_df.head(10))
else:
    print("\nNo data found! Check your DISH_ROI and TARGET_CLUSTER_ID settings.")

In [ ]:
# ============================================
# VISUALIZE EXTRACTED CLUSTER (CROPPED TO REGION)
# ============================================

if len(extracted_df) > 0:
    # Get cluster ID(s) in extracted data
    cluster_ids = extracted_df['cluster'].dropna().unique()
    cluster_id_display = f"{int(cluster_ids[0])}" if len(cluster_ids) == 1 else f"{[int(c) for c in cluster_ids]}"
    
    # Load reference image - works for both video and images
    ref_img_rgb = get_reference_frame(0)
    h, w = ref_img_rgb.shape[:2]
    
    # Determine region to show
    if DISH_ROI is not None:
        # Use specified ROI
        x_min, x_max = DISH_ROI['x_min'], DISH_ROI['x_max']
        y_min, y_max = DISH_ROI['y_min'], DISH_ROI['y_max']
    else:
        # Calculate bounding box from cluster data (with padding)
        pad = 100  # Padding around cluster
        x_min = max(0, int(extracted_df['centroid_x'].min() - pad))
        x_max = min(w, int(extracted_df['centroid_x'].max() + pad))
        y_min = max(0, int(extracted_df['centroid_y'].min() - pad))
        y_max = min(h, int(extracted_df['centroid_y'].max() + pad))
    
    # Crop image to region
    cropped_img = ref_img_rgb[y_min:y_max, x_min:x_max]
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    ax.imshow(cropped_img)
    
    # Plot extracted cluster(s) on first frame (adjust coordinates for cropped view)
    frame_0_data = extracted_df[extracted_df['frame'] == 0].copy()
    if len(frame_0_data) > 0:
        # Adjust coordinates relative to cropped region
        frame_0_data = frame_0_data.copy()
        frame_0_data['centroid_x_cropped'] = frame_0_data['centroid_x'] - x_min
        frame_0_data['centroid_y_cropped'] = frame_0_data['centroid_y'] - y_min
        
        colors = plt.cm.Set3(np.linspace(0, 1, len(cluster_ids)))
        for idx, cluster_id in enumerate(cluster_ids):
            cluster_frame_data = frame_0_data[frame_0_data['cluster'] == cluster_id]
            ax.scatter(cluster_frame_data['centroid_x_cropped'], cluster_frame_data['centroid_y_cropped'],
                      c=[colors[idx]], s=100, alpha=0.8, edgecolors='black', linewidths=2,
                      label=f'Cluster {int(cluster_id)} ({cluster_frame_data["track_id"].nunique()} tracks)')
        
        # Plot trajectories of tracks in this cluster (adjusted for cropped view)
        for cluster_id in cluster_ids:
            cluster_tracks = extracted_df[extracted_df['cluster'] == cluster_id]['track_id'].unique()
            for tid in cluster_tracks[:20]:  # Limit to first 20 tracks for clarity
                track_data = extracted_df[extracted_df['track_id'] == tid].sort_values('frame').copy()
                if len(track_data) > 1:
                    # Adjust coordinates relative to cropped region
                    track_x = track_data['centroid_x'].values - x_min
                    track_y = track_data['centroid_y'].values - y_min
                    ax.plot(track_x, track_y, '-', linewidth=1, alpha=0.3, color='gray')
    
    # Set title
    if DISH_ROI is not None:
        title = f'Cluster {cluster_id_display} in Selected Dish Region\n(cropped view)'
    else:
        title = f'Cluster {cluster_id_display}\n(cropped to cluster region)'
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Set axis labels with original coordinates
    ax.set_xlabel(f'X (px)', fontsize=VIZ_AXIS_FONTSIZE)
    ax.set_ylabel(f'Y (px)', fontsize=VIZ_AXIS_FONTSIZE)
    ax.legend(loc='upper right', fontsize=VIZ_LEGEND_FONTSIZE)
    ax.tick_params(axis='both', which='major', labelsize=18)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontfamily('Arial')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nShowing region: x=[{x_min}, {x_max}], y=[{y_min}, {y_max}]")
    print(f"Cropped image size: {x_max-x_min} x {y_max-y_min} pixels")
    
    # Save extracted cluster data (local fallback if OneDrive times out)
    if TARGET_CLUSTER_ID is not None:
        save_path = OUTPUT_DIR + f"cluster_{int(TARGET_CLUSTER_ID)}_extracted.csv"
        try:
            extracted_df.to_csv(save_path, index=False)
        except (TimeoutError, OSError) as e:
            local_dir = os.path.join(os.getcwd(), 'paper_analysis_output')
            os.makedirs(local_dir, exist_ok=True)
            save_path = os.path.join(local_dir, f"cluster_{int(TARGET_CLUSTER_ID)}_extracted.csv")
            extracted_df.to_csv(save_path, index=False)
            print(f"OneDrive save failed ({type(e).__name__}), saved locally instead.")
        print(f"\nSaved extracted cluster data to: {save_path}")
else:
    print("No data to visualize. Please check your extraction parameters.")

## 4.3 Generate Timeline GIF
Create animated GIF showing cluster evolution over time.

In [ ]:
# ============================================
# GENERATE CLUSTER TIMELINE GIF (SEGMENTATION OVERLAY STYLE)
# ============================================
# Same visual style as 3.1c segmentation video:
#   - Grayscale background
#   - Semi-transparent colored instance masks (cluster fronds only)
#   - Yellow numbered labels with dark shadow
#   - Compact HUD box: Fronds count only (green)
# Cropped to the cluster's ROI.

import imageio
from pathlib import Path
from scipy.spatial.distance import cdist
import gc

if len(extracted_df) > 0:
    print("=" * 60)
    print("GENERATING CLUSTER TIMELINE GIF (Segmentation Overlay Style)")
    print("=" * 60)

    # --- cluster metadata ---
    cluster_ids = extracted_df['cluster'].dropna().unique()
    cluster_id_display = (
        f"{int(cluster_ids[0])}" if len(cluster_ids) == 1
        else f"{[int(c) for c in cluster_ids]}"
    )

    # --- crop region ---
    ref_img_rgb = get_reference_frame(0)
    h, w = ref_img_rgb.shape[:2]
    del ref_img_rgb
    gc.collect()

    if DISH_ROI is not None:
        x_min, x_max = DISH_ROI['x_min'], DISH_ROI['x_max']
        y_min, y_max = DISH_ROI['y_min'], DISH_ROI['y_max']
    else:
        pad = 100
        x_min = max(0, int(extracted_df['centroid_x'].min() - pad))
        x_max = min(w, int(extracted_df['centroid_x'].max() + pad))
        y_min = max(0, int(extracted_df['centroid_y'].min() - pad))
        y_max = min(h, int(extracted_df['centroid_y'].max() + pad))

    all_frames = sorted(extracted_df['frame'].unique())
    print(f"\nCluster {cluster_id_display}: {len(all_frames)} frames")
    print(f"Region: x=[{x_min}, {x_max}], y=[{y_min}, {y_max}]")
    print(f"Cropped size: {x_max - x_min} x {y_max - y_min} pixels")

    # --- consistent track-ID → color map (HSV, same palette as 3.1c) ---
    track_ids_all = sorted(extracted_df['track_id'].dropna().unique())
    track_colors = {}
    for idx, tid in enumerate(track_ids_all, start=1):
        hue = int((idx * 37) % 180)
        hsv = np.uint8([[[hue, 220, 230]]])
        rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)[0][0]
        track_colors[int(tid)] = tuple(int(c) for c in rgb)

    # --- GIF output path ---
    if "OneDrive" in OUTPUT_DIR or "CloudStorage" in OUTPUT_DIR:
        gif_dir = os.path.join(os.getcwd(), "paper_analysis_gifs")
        os.makedirs(gif_dir, exist_ok=True)
        gif_path = os.path.join(gif_dir, f"cluster_{cluster_id_display}_timeline.gif")
    else:
        gif_path = OUTPUT_DIR + f"cluster_{cluster_id_display}_timeline.gif"

    CHUNK_SIZE = 50
    ALPHA = 0.55
    writer = imageio.get_writer(gif_path, mode='I', fps=10, loop=0)

    try:
        for chunk_start in range(0, len(all_frames), CHUNK_SIZE):
            chunk_end = min(chunk_start + CHUNK_SIZE, len(all_frames))
            chunk_frames = all_frames[chunk_start:chunk_end]
            print(f"\nChunk {chunk_start // CHUNK_SIZE + 1}/"
                  f"{(len(all_frames) - 1) // CHUNK_SIZE + 1}  "
                  f"(frames {chunk_start + 1}-{chunk_end} of {len(all_frames)})...")

            for frame_num in chunk_frames:
                frame_data = extracted_df[extracted_df['frame'] == frame_num]
                if len(frame_data) == 0:
                    continue

                # --- load frame ---
                if USE_VIDEO:
                    cap = cv2.VideoCapture(VIDEO_PATH)
                    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
                    ret, img_bgr = cap.read()
                    cap.release()
                else:
                    img_file = frame_data['image_file'].iloc[0]
                    img_bgr = cv2.imread(str(Path(IMAGE_DIRECTORY) / img_file))

                frame_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                del img_bgr

                # --- run segmentation on the full frame ---
                instance_mask, _ = segment_boundary_model(frame_rgb, min_area=MIN_AREA)

                # --- match mask instances to cluster detections by centroid ---
                det_centroids = frame_data[['centroid_y', 'centroid_x']].values  # (N, 2)
                det_track_ids = frame_data['track_id'].values

                uid_list = [u for u in np.unique(instance_mask) if u > 0]
                mask_centroids = []
                for uid in uid_list:
                    ys, xs = np.where(instance_mask == uid)
                    mask_centroids.append([ys.mean(), xs.mean()])
                mask_centroids = np.array(mask_centroids) if mask_centroids else np.empty((0, 2))

                # For each detection, find closest mask instance (within 30 px)
                matched_uids = {}  # mask_uid -> track_id
                if len(mask_centroids) > 0 and len(det_centroids) > 0:
                    dists = cdist(det_centroids, mask_centroids)
                    for det_idx in range(len(det_centroids)):
                        closest_mask_idx = dists[det_idx].argmin()
                        if dists[det_idx, closest_mask_idx] < 30:
                            matched_uids[uid_list[closest_mask_idx]] = int(det_track_ids[det_idx])

                # --- build grayscale canvas (full frame) ---
                gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
                canvas = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
                del frame_rgb, gray

                # --- draw colored masks only for matched (cluster) fronds ---
                num_fronds = len(matched_uids)
                for uid, tid in matched_uids.items():
                    mask = (instance_mask == uid)
                    c = np.array(track_colors.get(tid, (200, 200, 200)), dtype=np.float32)

                    roi = canvas[mask].astype(np.float32)
                    canvas[mask] = np.clip(
                        (1 - ALPHA) * roi + ALPHA * c, 0, 255
                    ).astype(np.uint8)

                    # Yellow numbered label (track ID) with shadow
                    ys, xs = np.where(mask)
                    cx, cy = int(xs.mean()), int(ys.mean())
                    label = str(tid)
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    cv2.putText(canvas, label, (cx - 1, cy + 1), font, 0.7,
                                (0, 0, 0), 3, cv2.LINE_AA)
                    cv2.putText(canvas, label, (cx, cy), font, 0.7,
                                (0, 255, 255), 2, cv2.LINE_AA)

                del instance_mask

                # --- crop to cluster ROI ---
                cropped = canvas[y_min:y_max, x_min:x_max].copy()
                del canvas

                # --- HUD (top-left, compact - Fronds only, matches Cluster badge size) ---
                frond_text = f"Fronds: {num_fronds}"
                font = cv2.FONT_HERSHEY_SIMPLEX
                fs, th = 0.6, 2
                (tw, _th), _ = cv2.getTextSize(frond_text, font, fs, th)
                box_w = tw + 16
                box_h = 34
                cv2.rectangle(cropped, (0, 0), (box_w, box_h), (0, 0, 0), -1)
                cv2.putText(cropped, frond_text, (8, 24),
                            font, fs, (0, 255, 0), th, cv2.LINE_AA)

                # --- cluster ID badge (top-right) ---
                cl_text = f"Cluster {cluster_id_display}"
                (tw, _th), _ = cv2.getTextSize(cl_text, font, 0.6, 2)
                cl_x = cropped.shape[1] - tw - 15
                cv2.rectangle(cropped, (cl_x - 8, 0),
                              (cropped.shape[1], 34), (0, 0, 0), -1)
                cv2.putText(cropped, cl_text, (cl_x, 24),
                            font, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

                writer.append_data(cropped)
                del cropped

            gc.collect()
            print(f"  Chunk complete. Memory cleared.")

    finally:
        writer.close()

    print(f"\n{'=' * 60}")
    print("GIF COMPLETE!")
    print(f"{'=' * 60}")
    print(f"Saved: {gif_path}")
    print(f"Frames: {len(all_frames)}")
    print(f"Duration: {len(all_frames) / 10:.1f} seconds at 10 fps")
    print(f"Time span: "
          f"{all_frames[0] * MINUTES_PER_FRAME / 60:.1f}h - "
          f"{all_frames[-1] * MINUTES_PER_FRAME / 60:.1f}h")
    print(f"{'=' * 60}\n")

else:
    print("No extracted cluster data found. Run the extraction cell first.")

## 4.4 Filter False Budding Events
Remove segmentation artifacts that cause false budding events. Produces `population_per_frame_filtered` for the population plot in 5.1.

In [ ]:
# ============================================
# 4.4 FILTER FALSE DIVISIONS (merged)
# ============================================
# Removes segmentation artifacts that cause false budding events.
# Output: population_per_frame_filtered (used by 5.1 Population Plot)

if len(extracted_df) > 0:
    from scipy.signal import medfilt
    
    # Count individuals per frame
    population_per_frame = extracted_df.groupby('frame').size().reset_index(name='count')
    population_per_frame['hours'] = population_per_frame['frame'] * MINUTES_PER_FRAME / 60
    population_per_frame_raw = population_per_frame.copy()
    
    # Median filter (removes spikes)
    window_size = 5
    population_per_frame_raw['count_median'] = medfilt(population_per_frame_raw['count'], kernel_size=window_size)
    
    # Require budding to persist for at least N frames (real budding events last 2-3 frames)
    MIN_PERSISTENCE = 2
    population_per_frame_raw['count_persistent'] = population_per_frame_raw['count'].copy()
    
    for i in range(len(population_per_frame_raw) - MIN_PERSISTENCE):
        current_count = population_per_frame_raw.iloc[i]['count']
        future_counts = population_per_frame_raw.iloc[i+1:i+MIN_PERSISTENCE+1]['count'].values
        if current_count > (population_per_frame_raw.iloc[i-1]['count'] if i > 0 else current_count):
            if all(fc < current_count for fc in future_counts[:2]):
                if i > 0:
                    population_per_frame_raw.loc[population_per_frame_raw.index[i], 'count_persistent'] = \
                        population_per_frame_raw.iloc[i-1]['count']
    
    # Build smoothed dataframe
    population_smoothed = population_per_frame_raw[['frame', 'hours']].copy()
    population_smoothed['count_raw'] = population_per_frame_raw['count']
    population_smoothed['count_median'] = population_per_frame_raw['count_median']
    population_smoothed['count_persistent'] = population_per_frame_raw['count_persistent']
    
    # Enforce monotonic increase (population can only increase or stay same)
    count_monotonic = population_smoothed['count_persistent'].copy()
    max_so_far = count_monotonic.iloc[0]
    for i in range(len(count_monotonic)):
        if count_monotonic.iloc[i] < max_so_far:
            count_monotonic.iloc[i] = max_so_far
        else:
            max_so_far = count_monotonic.iloc[i]
    population_smoothed['count_smoothed'] = count_monotonic
    
    population_per_frame_filtered = population_smoothed.copy()
    
    print(f"Filtered false budding events: {population_smoothed['count_raw'].min()}-{population_smoothed['count_raw'].max()} -> smoothed {population_smoothed['count_smoothed'].min()}-{population_smoothed['count_smoothed'].max()}")
    print(f"Fluctuations removed: {(population_smoothed['count_raw'] != population_smoothed['count_smoothed']).sum()}")
    
    # Save
    save_path = OUTPUT_DIR + f"cluster_{extracted_df['cluster'].dropna().iloc[0]:.0f}_population_smoothed.csv"
    population_smoothed.to_csv(save_path, index=False)
    print(f"Saved: {save_path}")
else:
    print("No extracted cluster data. Run extraction cell (4.2/4.3) first.")

---
# PART 5: VISUALIZATIONS
---
Generate publication-ready figures.

## 5.1 Population Plot
Smoothed population count over time with day/night shading.

In [ ]:
# ============================================
# PUBLICATION-READY SMOOTHED POPULATION PLOT
# ============================================
# Shows raw fluctuations faded in background, smoothed data prominent
# Day/night shading alternates every 24 hours

if len(extracted_df) > 0 and 'population_per_frame_filtered' in locals():
    # Set font to Arial (or Helvetica as fallback) and increase size
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = 22
    plt.rcParams['axes.labelsize'] = 24
    plt.rcParams['axes.titlesize'] = 26
    plt.rcParams['xtick.labelsize'] = 20
    plt.rcParams['ytick.labelsize'] = 20
    plt.rcParams['legend.fontsize'] = 22

    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))

    # Round to integers (population must be whole numbers)
    count_smoothed = np.round(population_per_frame_filtered['count_smoothed']).astype(int)
    
    # Get raw data (before smoothing) if available
    if 'count_raw' in population_per_frame_filtered.columns:
        count_raw = population_per_frame_filtered['count_raw'].astype(int)
    elif 'count' in population_per_frame_filtered.columns:
        count_raw = population_per_frame_filtered['count'].astype(int)
    else:
        count_raw = None

    # =========================================================
    # PLOT RAW FLUCTUATIONS FADED IN BACKGROUND
    # =========================================================
    if count_raw is not None:
        ax.step(population_per_frame_filtered['hours'],
               count_raw,
               where='post', linewidth=1.5, color='#228B22', alpha=0.25,
               label='Raw counts')

    # =========================================================
    # PLOT SMOOTHED DATA (PROMINENT)
    # =========================================================
    ax.step(population_per_frame_filtered['hours'],
           count_smoothed,
           where='post', linewidth=2.5, color='#228B22', label='Smoothed (5h window)')

    # Axis labels
    ax.set_xlabel('Time (h)', fontsize=20, fontweight='bold')
    ax.set_ylabel('Number of Plants', fontsize=20, fontweight='bold')

    # Force integer y-axis and add padding at top for labels
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    y_min, y_max = ax.get_ylim()
    ax.set_ylim(y_min, y_max + 1)  # Add 1 unit of padding at top

    # Calculate doubling time (all plants + budding lineage)
    from scipy.optimize import curve_fit

    def exponential_growth(t, N0, r):
        return N0 * np.exp(r * t)

    doubling_time = None
    doubling_time_budding = None
    try:
        popt, _ = curve_fit(exponential_growth,
                           population_per_frame_filtered['hours'].values,
                           count_smoothed.values,
                           p0=[count_smoothed.iloc[0], 0.01],
                           maxfev=5000)
        if popt[1] > 0:
            doubling_time = np.log(2) / popt[1]
    except:
        pass

    # Budding-only doubling time (if the budding vs non-budding cell has been run)
    if 'results' in dir() and 'Budding only' in results:
        doubling_time_budding = results['Budding lineage']['doubling_time']

    # =========================================================
    # DAY/NIGHT SHADING - Every 24h cycle has:
    #   - First 12h = Night (gray shading)
    #   - Second 12h = Day (no shading)
    # =========================================================
    max_hours = population_per_frame_filtered['hours'].max()

    # Add night shading for first 12h of each 24h cycle
    for period_start in np.arange(0, max_hours + 24, 24):
        if period_start <= max_hours:
            night_end = min(period_start + 12, max_hours)
            ax.axvspan(period_start, night_end, alpha=0.15, color='gray', zorder=0)

    # =========================================================
    # DAY/NIGHT LABELS - Inside containers at top (y=0.92 in axes coords)
    # =========================================================
    for period_start in np.arange(0, max_hours, 24):
        # Night label (centered at period_start + 6)
        night_center = period_start + 6
        if night_center <= max_hours:
            ax.text(night_center, 0.97, 'Night',
                   ha='center', va='top', fontsize=14, style='italic', color='gray',
                   transform=ax.get_xaxis_transform())

        # Day label (centered at period_start + 18)
        day_center = period_start + 18
        if day_center <= max_hours:
            ax.text(day_center, 0.97, 'Day',
                   ha='center', va='top', fontsize=14, style='italic', color='gray',
                   transform=ax.get_xaxis_transform())

    # Add doubling time box (above the plot, centered)
    if doubling_time:
        if doubling_time_budding:
            textstr = f'Doubling Time: {doubling_time:.1f} h (all)  |  {doubling_time_budding:.1f} h (budding lineage)'
        else:
            textstr = f'Doubling Time = {doubling_time:.1f} h (all plants)'
        props = dict(boxstyle='square', facecolor='white', edgecolor='black', linewidth=1.5)
        fig.text(0.5, 0.96, textstr, transform=fig.transFigure, fontsize=16, fontweight='bold',
                 ha='center', va='top', bbox=props)

    # Remove grid and clean up spines
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)

    plt.tight_layout(rect=[0, 0, 1, 0.92])

    # Save figure
    cluster_id = extracted_df['cluster'].dropna().iloc[0]
    fig_path = OUTPUT_DIR + f"cluster_{int(cluster_id)}_population_publication.png"
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved publication-ready figure: {fig_path}")

    plt.show()

    # Reset font settings
    plt.rcParams['font.size'] = 10
    plt.rcParams['axes.labelsize'] = 10
    plt.rcParams['axes.titlesize'] = 10
    plt.rcParams['xtick.labelsize'] = 10
    plt.rcParams['ytick.labelsize'] = 10

    # Print summary
    print(f"\nPopulation range: {count_smoothed.iloc[0]} → {count_smoothed.iloc[-1]} plants")
    if doubling_time:
        print(f"Doubling time (all): {doubling_time:.1f} hours")
    if doubling_time_budding:
        print(f"Doubling time (budding lineage): {doubling_time_budding:.1f} hours")
    if count_raw is not None:
        print(f"Raw fluctuations shown faded (alpha=0.25) behind smoothed line")
else:
    print("Please run the previous cells first.")


## 5.2 Area Over Time
Individual cell growth trajectories.

**What This Plot Shows:**
1. **Individual Growth**: Each colored line = one duckweed cell's area (mm²) over time
2. **Tracking Validation**: Smooth lines = tracking works correctly (no jumps or breaks)
3. **Budding Events**: New lines appearing = new daughter cells appearing after budding
4. **Growth Patterns**: 
   - **Mothers (Gen 0)**: Larger, more stable areas (0.02-0.045 mm²)
   - **Daughters (Gen 1+)**: Start small, grow steadily (0.005-0.03 mm²)
5. **Biological Reality**: Area changes reflect actual cell growth and budding

**Why It Matters:**
- Shows tracking is working (continuous trajectories)
- Links population increases (from population plot) to actual budding events
- Reveals growth rates and patterns for different generations
- Validates that area measurements match what we see visually

**Filtering:**
This plot uses **FILTERED** data - false budding detections (segmentation artifacts) are removed to match the smoothed population plot.

In [ ]:
# ============================================
# AREA OVER TIME - INDIVIDUAL CELL GROWTH (FILTERED)
# ============================================
# Shows how individual cells grow over time in the selected cluster
# Uses FILTERED data (false budding events removed) to match population plot
# Demonstrates tracking works and reveals growth patterns

if len(extracted_df) > 0:
    print("="*60)
    print("INDIVIDUAL CELL AREA OVER TIME (FILTERED)")
    print("="*60)
    
    # Apply filtering directly using smoothed population data
    use_filtered = False
    working_df = extracted_df.copy()
    
    if 'population_per_frame_filtered' in locals():
        print("✓ Smoothed population data found. Filtering tracking data...")
        
        # Create mapping: frame -> expected count (from smoothed data)
        frame_to_expected = dict(zip(
            population_per_frame_filtered['frame'],
            population_per_frame_filtered['count_smoothed'].astype(int)
        ))
        frame_to_raw = dict(zip(
            population_per_frame_filtered['frame'],
            population_per_frame_filtered['count_raw'].astype(int)
        ))
        
        # Filter detections: remove false positives where raw > expected
        filtered_df = working_df.copy()
        removed_count = 0
        
        for frame_num in working_df['frame'].unique():
            if frame_num in frame_to_expected and frame_num in frame_to_raw:
                raw_count = frame_to_raw[frame_num]
                expected_count = frame_to_expected[frame_num]
                
                if raw_count > expected_count:
                    # This frame has false detections - remove excess
                    frame_data = filtered_df[filtered_df['frame'] == frame_num]
                    
                    # Count detections per track (for filtering strategy)
                    track_counts = {}
                    for tid in frame_data['track_id'].dropna().unique():
                        track_counts[tid] = len(filtered_df[filtered_df['track_id'] == tid])
                    
                    # Strategy: Keep tracks with most detections (longest-lived tracks)
                    # These are more likely to be real, persistent tracks
                    sorted_tracks = sorted(track_counts.items(), key=lambda x: x[1], reverse=True)
                    tracks_to_keep = [tid for tid, _ in sorted_tracks[:expected_count]]
                    
                    # Remove detections from tracks not in keep list
                    mask_to_remove = (filtered_df['frame'] == frame_num) & \
                                    (~filtered_df['track_id'].isin(tracks_to_keep))
                    removed_count += mask_to_remove.sum()
                    filtered_df = filtered_df[~mask_to_remove]
        
        if removed_count > 0:
            working_df = filtered_df
            use_filtered = True
            frames_filtered = len([f for f in extracted_df['frame'].unique() 
                                  if f in frame_to_expected and frame_to_raw.get(f, 0) > frame_to_expected.get(f, 0)])
            print(f"✓ Filtered {removed_count} false detections across {frames_filtered} frames")
        else:
            print("✓ No false detections to filter (data already matches smoothed population)")
            use_filtered = True
    else:
        print("⚠️  No smoothed population data found!")
        print("   Run Cell 37 first to generate filtered population data.")
        print("   Using raw (unfiltered) data - false budding events may appear.")
    
    cluster_id = working_df['cluster'].dropna().iloc[0]
    
    # Get all unique tracks in this cluster (from filtered data)
    cluster_tracks = working_df['track_id'].dropna().unique()
    
    print(f"\nCluster {int(cluster_id)} Analysis:")
    print(f"  Total tracks: {len(cluster_tracks)}")
    print(f"  Total detections: {len(working_df)}")
    print(f"  Frames covered: {working_df['frame'].nunique()}")
    
    if use_filtered:
        print(f"\n✓ Using FILTERED data - false budding events removed")
    else:
        print(f"\n⚠️  Using RAW data - may contain false budding events")
    
    # Set publication-ready fonts (match 5.1 population plot)
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = 20
    plt.rcParams['axes.labelsize'] = 20
    plt.rcParams['axes.titlesize'] = 24
    plt.rcParams['xtick.labelsize'] = 20
    plt.rcParams['ytick.labelsize'] = 20
    plt.rcParams['legend.fontsize'] = 18
    
    # Create figure (enlarged)
    fig, ax = plt.subplots(figsize=(18, 10))
    
    # Generate colors for tracks (consistent) - store globally for use in crop images
    np.random.seed(42)
    colors = plt.cm.tab20(np.linspace(0, 1, len(cluster_tracks)))
    # Create color map: track_id -> matplotlib color (0-1 RGB)
    track_color_map = {tid: colors[i] for i, tid in enumerate(cluster_tracks)}
    
    # Store for use in crop image visualization (Cell 43)
    globals()['track_color_map_global'] = track_color_map
    
    # Smooth area data to reduce noise from segmentation
    from scipy.signal import savgol_filter
    from scipy.ndimage import uniform_filter1d
    
    # Plot each track's area over time (from filtered data) with smoothing
    plotted_tracks = []
    for idx, track_id in enumerate(cluster_tracks):
        track_data = working_df[working_df['track_id'] == track_id].sort_values('frame').copy()
        
        if len(track_data) > 1:  # Need at least 2 points to plot
            hours = track_data['hours'].values
            areas = track_data['area_mm2'].values
            
            # Smooth the area measurements to reduce segmentation noise
            if len(areas) >= 5:
                # Use Savitzky-Golay filter (preserves trends better than moving average)
                try:
                    window_length = min(5, len(areas) if len(areas) % 2 == 1 else len(areas) - 1)
                    if window_length >= 3:
                        areas_smooth = savgol_filter(areas, window_length, 2)  # polynomial order 2
                        areas_smooth = np.maximum(areas_smooth, 0.001)  # Ensure non-negative
                    else:
                        areas_smooth = areas
                except:
                    # Fallback to uniform filter if Savitzky-Golay fails
                    areas_smooth = uniform_filter1d(areas, size=3, mode='nearest')
            elif len(areas) >= 3:
                # Simple moving average for short tracks
                areas_smooth = uniform_filter1d(areas, size=min(3, len(areas)), mode='nearest')
            else:
                areas_smooth = areas
            
            color = track_color_map[track_id]  # Use color from map
            ax.plot(hours, areas_smooth, 
                   linewidth=2.5, alpha=0.85, color=color, 
                   label=f'Track {int(track_id)}', zorder=5)
            plotted_tracks.append(track_id)
    
    # Add day/night shading (24-hour cycles, assuming 8PM start)
    max_hours = working_df['hours'].max()
    day_night_periods = np.arange(0, max_hours + 24, 24)
    
    # Shade night periods (first 12h of each 24h cycle) - no legend entry (labeled in plot)
    for i, period in enumerate(day_night_periods[:-1]):
        if period < max_hours:
            night_end = min(period + 12, max_hours)
            ax.axvspan(period, night_end, alpha=0.15, color='gray', zorder=0)
    
    # Day periods have no shading (white background) - no legend entry needed
    
    # Add day/night labels inside plot (like 5.1 population plot)
    for i, period in enumerate(day_night_periods[:-1]):
        if period <= max_hours:
            night_center = period + 6
            day_center = period + 18
            if night_center <= max_hours:
                ax.text(night_center, 0.97, 'Night',
                       ha='center', va='top', fontsize=14, style='italic', color='gray',
                       transform=ax.get_xaxis_transform())
            if day_center <= max_hours:
                ax.text(day_center, 0.97, 'Day',
                       ha='center', va='top', fontsize=14, style='italic', color='gray',
                       transform=ax.get_xaxis_transform())
    
    # Formatting - cleaner styling
    filter_label = " (FILTERED)" if use_filtered else " (RAW)"
    ax.set_xlabel('Time (hours)', fontsize=48, fontweight='bold')
    ax.set_ylabel('Area (mm²)', fontsize=48, fontweight='bold')
    ax.set_title(f'Individual Cell Growth - Cluster {int(cluster_id)}{filter_label}\n(Smoothed area over time for each tracked cell)', 
                fontsize=VIZ_TITLE_FONTSIZE, fontweight='bold', pad=20)
    
    # Improve axis limits for better visualization
    ax.set_ylim(bottom=0)  # Start y-axis at 0
    
    # Legend (limit to first 10 tracks to avoid clutter)
    if len(plotted_tracks) <= 15:
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=18, 
                 title='Track ID', title_fontsize=20)
    else:
        ax.text(0.02, 0.98, f'{len(plotted_tracks)} tracks shown', 
               transform=ax.transAxes, fontsize=18, verticalalignment='top', fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Remove grid
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    
    # Tick labels (doubled)
    ax.tick_params(axis='both', which='major', labelsize=40)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = OUTPUT_DIR + f"cluster_{int(cluster_id)}_area_over_time.png"
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved: {fig_path}")
    
    plt.show()
    
    # Print summary statistics
    print(f"\n{'='*60}")
    print("AREA GROWTH SUMMARY")
    print(f"{'='*60}")
    print(f"Tracks plotted: {len(plotted_tracks)}")
    
    # Calculate growth statistics for each track
    growth_stats = []
    for track_id in plotted_tracks[:10]:  # Show first 10 for summary
        track_data = working_df[working_df['track_id'] == track_id].sort_values('frame')
        if len(track_data) > 1:
            initial_area = track_data['area_mm2'].iloc[0]
            final_area = track_data['area_mm2'].iloc[-1]
            time_span = track_data['hours'].iloc[-1] - track_data['hours'].iloc[0]
            growth_rate = (final_area - initial_area) / time_span if time_span > 0 else 0
            
            growth_stats.append({
                'track_id': int(track_id),
                'initial_area': initial_area,
                'final_area': final_area,
                'time_span': time_span,
                'growth_rate': growth_rate
            })
    
    if len(growth_stats) > 0:
        stats_df = pd.DataFrame(growth_stats)
        print("\nSample track statistics (first 10):")
        print(stats_df.to_string(index=False))
    
    # Reset font settings
    plt.rcParams['font.size'] = 10
    plt.rcParams['axes.labelsize'] = 10
    
else:
    print("No extracted cluster data found. Please run Cell 31 first to extract a cluster.")

## 5.3 Area with Crop Images
Area plot with cropped images showing actual cells.

In [ ]:
# ============================================
# AREA OVER TIME WITH CROP IMAGES - IMAGES ON TOP
# ============================================
# Legend at TOP under title, horizontal layout
# PUBLICATION READY: Arial font, 2x size

if len(extracted_df) > 0:
    # Set publication-ready fonts - 2X size, BOLD
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = 20
    plt.rcParams['font.weight'] = 'bold'
    plt.rcParams['axes.labelsize'] = 24
    plt.rcParams['axes.labelweight'] = 'bold'
    plt.rcParams['axes.titlesize'] = 26
    plt.rcParams['axes.titleweight'] = 'bold'
    plt.rcParams['xtick.labelsize'] = 20
    plt.rcParams['ytick.labelsize'] = 20
    plt.rcParams['legend.fontsize'] = 18
    # Use filtered data if available
    if 'extracted_df_filtered' in locals() and extracted_df_filtered is not None and len(extracted_df_filtered) > 0:
        use_filtered = True
        working_df = extracted_df_filtered.copy()
        print("Using FILTERED data (false budding events removed)")
    else:
        use_filtered = False
        working_df = extracted_df.copy()
        print("⚠️  Using raw data.")
    
    cluster_id = working_df['cluster'].dropna().iloc[0]
    
    all_frames = sorted(working_df['frame'].unique())
    
    n_images = 6
    frame_indices = np.linspace(0, len(all_frames)-1, n_images, dtype=int)
    selected_frames = [all_frames[i] for i in frame_indices]
    
    # Load reference image - works for both video and images
    ref_img_rgb = get_reference_frame(0)
    h, w = ref_img_rgb.shape[:2]
    
    if DISH_ROI is not None:
        x_min, x_max = DISH_ROI['x_min'], DISH_ROI['x_max']
        y_min, y_max = DISH_ROI['y_min'], DISH_ROI['y_max']
    else:
        pad = 100
        x_min = max(0, int(working_df['centroid_x'].min() - pad))
        x_max = min(w, int(working_df['centroid_x'].max() + pad))
        y_min = max(0, int(working_df['centroid_y'].min() - pad))
        y_max = min(h, int(working_df['centroid_y'].max() + pad))
    
    # Create figure with extra space at top for legend
    fig = plt.figure(figsize=(16, 11))
    gs = gridspec.GridSpec(2, n_images, height_ratios=[1, 2], hspace=0.08, wspace=0.3,
                          top=0.88)  # Leave space at top for legend
    
    cluster_tracks = list(working_df['track_id'].dropna().unique())
    
    # Use distinct, vibrant colors for tracks
    distinct_colors = [
        '#1f77b4',  # Blue
        '#ff7f0e',  # Orange
        '#2ca02c',  # Green
        '#d62728',  # Red (different from dashed lines)
        '#9467bd',  # Purple
        '#8c564b',  # Brown
        '#5c7cfa',  # Indigo
        '#7f7f7f',  # Gray
        '#bcbd22',  # Yellow-green
        '#17becf',  # Cyan
        '#c0392b',  # Coral
        '#4ecdc4',  # Teal
        '#45b7d1',  # Sky blue
        '#96ceb4',  # Sage
        '#ffeaa7',  # Cream
        '#dfe6e9',  # Light gray
    ]
    
    color_map = {}
    for i, tid in enumerate(cluster_tracks):
        color_map[tid] = distinct_colors[i % len(distinct_colors)]
    
    # Pre-compute which tracks will actually be plotted (pass length filter AND appear in selected frames)
    MIN_TRACK_LENGTH = 10
    plotted_track_set = set()
    selected_frames_set = set(selected_frames)  # For fast lookup
    for tid in cluster_tracks:
        track_data = working_df[working_df['track_id'] == tid].sort_values('frame')
        if len(track_data) >= MIN_TRACK_LENGTH:
            # Only include tracks that appear in at least one of the selected image frames
            track_frames = set(track_data['frame'].unique())
            if track_frames.intersection(selected_frames_set):
                plotted_track_set.add(tid)
    
    # =========================================================
    # TOP ROW: Crop images
    # =========================================================
    for idx, frame_num in enumerate(selected_frames):
        ax_img = fig.add_subplot(gs[0, idx])
        
        frame_data = extracted_df[extracted_df['frame'] == frame_num]
        hours = frame_num * MINUTES_PER_FRAME / 60
        
        if len(frame_data) > 0:
            # Load frame - works for both video and images
            if USE_VIDEO:
                img_bgr = get_reference_frame(frame_num)
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            else:
                img_file = frame_data['image_file'].iloc[0]
                img_path = Path(IMAGE_DIRECTORY) / img_file
                img = cv2.imread(str(img_path))
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            cropped = img_rgb[y_min:y_max, x_min:x_max]
            
            gray = cv2.cvtColor(cropped, cv2.COLOR_RGB2GRAY)
            seg_frame = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB).astype(np.float32)
            
            for _, row in frame_data.iterrows():
                if pd.notna(row['track_id']):
                    track_id = int(row['track_id'])
                    if track_id not in plotted_track_set:
                        continue  # Skip tracks not shown in area plot
                    track_color_hex = color_map.get(track_id, '#808080')
                    # Convert hex to RGB (0-255)
                    track_color_hex = track_color_hex.lstrip('#')
                    mask_color_rgb = np.array([int(track_color_hex[i:i+2], 16) for i in (0, 2, 4)])
                    mask_color = mask_color_rgb.astype(np.float32)
                    border_color = tuple(mask_color.astype(int).tolist())
                    
                    x1 = int(row['bbox_minc'] - x_min)
                    y1 = int(row['bbox_minr'] - y_min)
                    x2 = int(row['bbox_maxc'] - x_min)
                    y2 = int(row['bbox_maxr'] - y_min)
                    
                    center_x = (x1 + x2) // 2
                    center_y = (y1 + y2) // 2
                    width = x2 - x1
                    height = y2 - y1
                    
                    x1 = max(0, min(x1, seg_frame.shape[1] - 1))
                    y1 = max(0, min(y1, seg_frame.shape[0] - 1))
                    x2 = max(0, min(x2, seg_frame.shape[1] - 1))
                    y2 = max(0, min(y2, seg_frame.shape[0] - 1))
                    center_x = max(0, min(center_x, seg_frame.shape[1] - 1))
                    center_y = max(0, min(center_y, seg_frame.shape[0] - 1))
                    
                    if x2 > x1 and y2 > y1 and width > 0 and height > 0:
                        mask = np.zeros((seg_frame.shape[0], seg_frame.shape[1]), dtype=np.uint8)
                        axes_length = (max(1, width // 2), max(1, height // 2))
                        cv2.ellipse(mask, (center_x, center_y), axes_length, 0, 0, 360, 255, -1)
                        
                        alpha = 0.6
                        mask_binary = (mask > 0).astype(np.float32)[:, :, np.newaxis]
                        for c in range(3):
                            seg_frame[:, :, c] = seg_frame[:, :, c] * (1 - alpha * mask_binary[:, :, 0]) + \
                                               mask_color[c] * (alpha * mask_binary[:, :, 0])
                        
                        seg_frame_uint = seg_frame.astype(np.uint8)
                        cv2.ellipse(seg_frame_uint, (center_x, center_y), axes_length, 0, 0, 360, border_color, 2)
                        
                        label = str(track_id)
                        font = cv2.FONT_HERSHEY_SIMPLEX
                        font_scale, thickness = 0.4, 1
                        (tw, th), _ = cv2.getTextSize(label, font, font_scale, thickness)
                        lx = center_x - tw // 2
                        ly = center_y + th // 2
                        
                        bg_color = tuple(mask_color.astype(int).tolist())
                        cv2.rectangle(seg_frame_uint, (lx - 2, ly - th - 2), (lx + tw + 2, ly + 2), bg_color, -1)
                        cv2.rectangle(seg_frame_uint, (lx - 2, ly - th - 2), (lx + tw + 2, ly + 2), border_color, 1)
                        cv2.putText(seg_frame_uint, label, (lx, ly), font, font_scale, (255, 255, 0), thickness)
                        seg_frame = seg_frame_uint.astype(np.float32)
            
            ax_img.imshow(seg_frame.astype(np.uint8))
            count = len(frame_data)
            plant_word = 'plant' if count == 1 else 'plants'
            ax_img.set_title(f't = {hours:.1f}h\n{count} {plant_word}', fontsize=VIZ_AXIS_FONTSIZE, fontweight='bold')
            ax_img.axis('off')
    
    # =========================================================
    # BOTTOM: Area plot
    # =========================================================
    ax_area = fig.add_subplot(gs[1, :])
    
    # Store all track lines for stacked legend
    all_track_lines = []
    plotted_track_ids = []  # Keep track of which tracks pass the filter
    
    # Use only tracks that appear in selected image frames (already filtered above)
    
    for i, track_id in enumerate(cluster_tracks):
        if track_id not in plotted_track_set:
            continue  # Skip tracks not in selected frames
        track_data = working_df[working_df['track_id'] == track_id].sort_values('frame')
        plotted_track_ids.append(track_id)
        hours = track_data['hours'].values
        
        window_hours = 5
        window_size_frames = int(window_hours * 60 / MINUTES_PER_FRAME)
        
        track_data_temp = track_data.copy()
        track_data_temp['area_smoothed'] = track_data_temp['area_mm2'].rolling(
            window=window_size_frames, 
            center=True, 
            min_periods=1
        ).mean()
        
        areas_smooth = track_data_temp['area_smoothed'].values
        areas_smooth = np.maximum(areas_smooth, 0.001)
        
        line, = ax_area.plot(hours, areas_smooth, linewidth=2.5, alpha=0.85, color=color_map[track_id])
        all_track_lines.append(line)
    
    # Red dashed lines for timepoints
    for frame_num in selected_frames:
        frame_data = working_df[working_df['frame'] == frame_num]
        if len(frame_data) > 0:
            hours = frame_data['hours'].iloc[0]
            ax_area.axvline(hours, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    # Day/night shading
    max_hours = working_df['hours'].max()
    for period_start in np.arange(0, max_hours + 24, 24):
        if period_start <= max_hours:
            night_end = min(period_start + 12, max_hours)
            ax_area.axvspan(period_start, night_end, alpha=0.15, color='gray', zorder=0)
    
    ax_area.set_xlabel('Time (hours)', fontsize=28, fontweight='bold')
    ax_area.set_ylabel('Area (mm²)', fontsize=28, fontweight='bold')
    ax_area.set_ylim(bottom=0)
    ax_area.grid(False)
    ax_area.spines['top'].set_visible(False)
    ax_area.spines['right'].set_visible(False)
    ax_area.tick_params(labelsize=20)
    
    # Make tick labels bold
    for label in ax_area.get_xticklabels() + ax_area.get_yticklabels():
        label.set_fontweight('bold')
    
    # =========================================================
    # LEGEND AT TOP - Stacked tracks with single label
    # =========================================================
    from matplotlib.lines import Line2D
    from matplotlib.legend_handler import HandlerBase
    
    # Custom handler for stacked lines with spacing
    class StackedLinesHandler(HandlerBase):
        def __init__(self, colors, **kwargs):
            self.colors = colors
            super().__init__(**kwargs)
        
        def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
            lines = []
            n = len(self.colors)
            line_spacing = 4  # pixels between lines
            total_height = n * line_spacing
            start_y = ydescent + (height - total_height) / 2 + line_spacing / 2
            for i, color in enumerate(self.colors):
                y = start_y + i * line_spacing
                line = Line2D([xdescent, xdescent + width], [y, y], 
                             color=color, linewidth=2, alpha=0.85)
                line.set_transform(trans)
                lines.append(line)
            return lines
    
    # Get colors of plotted tracks (up to 5 for legend) - only tracks that passed the filter
    track_colors = [color_map[tid] for tid in plotted_track_ids[:min(5, len(plotted_track_ids))]]
    
    print(f"Plotted {len(plotted_track_ids)} tracks (filtered out {len(cluster_tracks) - len(plotted_track_ids)} short tracks < {MIN_TRACK_LENGTH} frames)")
    
    # Create legend entries
    from matplotlib.patches import Patch
    
    stacked_handle = Line2D([], [])  # Dummy handle for stacked tracks
    red_line = Line2D([0], [0], color='red', linestyle='--', linewidth=2)
    night_patch = Patch(facecolor='gray', alpha=0.15, edgecolor='none')
    day_patch = Patch(facecolor='white', edgecolor='gray', linewidth=0.5)
    
    legend_handles = [stacked_handle, red_line, night_patch, day_patch]
    legend_labels = ['Tracks', 'Image timepoints', 'Night', 'Day']
    
    # Create horizontal legend at top of figure
    fig.legend(legend_handles, legend_labels, 
              loc='upper center', 
              bbox_to_anchor=(0.5, 0.95),
              ncol=4,
              fontsize=18,
              frameon=True,
              fancybox=True,
              shadow=False,
              edgecolor='gray',
              handler_map={stacked_handle: StackedLinesHandler(track_colors)})
    
    filter_label = " (FILTERED)" if use_filtered else " (RAW)"
    plt.suptitle(f'Cluster {int(cluster_id)}: Individual Cell Growth{filter_label}', 
                fontsize=VIZ_TITLE_FONTSIZE+4, fontweight='bold', y=0.99)
    
    # Save figure
    fig_path = OUTPUT_DIR + f"cluster_{int(cluster_id)}_area_with_crops_alt.png"
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved: {fig_path}")
    
    plt.show()
    
    # Reset font settings
    plt.rcParams['font.size'] = 10
    plt.rcParams['font.weight'] = 'normal'
    plt.rcParams['axes.labelsize'] = 10
    plt.rcParams['axes.labelweight'] = 'normal'
    plt.rcParams['axes.titlesize'] = 10
    plt.rcParams['axes.titleweight'] = 'normal'
    plt.rcParams['xtick.labelsize'] = 10
    plt.rcParams['ytick.labelsize'] = 10

else:
    print("No extracted cluster data found.")


## 5.3b Area with Crop Images — Consistent Canonical IDs
Same layout as 5.3, but colors and IDs are based on **canonical_track_id** so that
a single physical plant keeps one color across its entire lifetime, even if the
tracker assigned it multiple raw IDs.

In [ ]:
# ============================================
# AREA OVER TIME WITH CROP IMAGES - CANONICAL IDs
# ============================================
# Chains fragmented track IDs into continuous physical plants,
# then detects budding events.

if len(extracted_df) > 0:
    # --- Font setup (publication-ready) ---
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = 20
    plt.rcParams['font.weight'] = 'bold'
    plt.rcParams['axes.labelsize'] = 24
    plt.rcParams['axes.labelweight'] = 'bold'
    plt.rcParams['axes.titlesize'] = 26
    plt.rcParams['axes.titleweight'] = 'bold'
    plt.rcParams['xtick.labelsize'] = 20
    plt.rcParams['ytick.labelsize'] = 20
    plt.rcParams['legend.fontsize'] = 18

    # --- Choose filtered vs raw data ---
    if 'extracted_df_filtered' in locals() and extracted_df_filtered is not None and len(extracted_df_filtered) > 0:
        use_filtered = True
        working_df = extracted_df_filtered.copy()
        print("Using FILTERED data (false budding events removed)")
    else:
        use_filtered = False
        working_df = extracted_df.copy()
        print("Using raw data.")

    # --- Ensure canonical_track_id exists ---
    if 'canonical_track_id' not in working_df.columns:
        if 'track_mapping' in dir() and track_mapping is not None:
            working_df['canonical_track_id'] = working_df['track_id'].map(track_mapping)
        else:
            working_df['canonical_track_id'] = working_df['track_id']
    working_df = working_df.dropna(subset=['canonical_track_id']).copy()
    working_df['canonical_track_id'] = working_df['canonical_track_id'].astype(int)

    if 'canonical_track_id' not in extracted_df.columns:
        if 'track_mapping' in dir() and track_mapping is not None:
            extracted_df['canonical_track_id'] = extracted_df['track_id'].map(track_mapping)
        else:
            extracted_df['canonical_track_id'] = extracted_df['track_id']

    # ==========================================================
    # LOCAL RE-STITCHING — chain fragments into physical plants
    # ==========================================================
    # Key rule: if a predecessor STOPS shortly after a successor starts
    # (brief overlap = tracker handoff), it's a continuation.
    # If the predecessor KEEPS GOING long after, it's a real budding event.
    LOCAL_MAX_GAP_FRAMES = 100
    LOCAL_MAX_STITCH_DIST = 120
    # Max frames the predecessor can CONTINUE after successor starts
    # and still be considered a handoff (not a budding event)
    HANDOFF_OVERLAP_LIMIT = 40

    # Build per-canonical-track summary
    can_summaries = []
    for cid, group in working_df.groupby('canonical_track_id'):
        sg = group.sort_values('frame')
        can_summaries.append({
            'cid': int(cid),
            'start_frame': int(sg['frame'].iloc[0]),
            'end_frame': int(sg['frame'].iloc[-1]),
            'start_x': float(sg['centroid_x'].iloc[0]),
            'start_y': float(sg['centroid_y'].iloc[0]),
            'end_x': float(sg['centroid_x'].iloc[-1]),
            'end_y': float(sg['centroid_y'].iloc[-1]),
            'mean_area': float(sg['area_mm2'].mean()),
            'n_frames': len(sg),
        })
    can_info_df = pd.DataFrame(can_summaries).sort_values('start_frame').reset_index(drop=True)

    print(f"Canonical fragments to stitch: {len(can_info_df)}")
    for _, r in can_info_df.iterrows():
        h_s = r['start_frame'] * MINUTES_PER_FRAME / 60
        h_e = r['end_frame'] * MINUTES_PER_FRAME / 60
        print(f"  cid={int(r['cid']):>3d}  t={h_s:5.1f}-{h_e:5.1f}h  "
              f"area={r['mean_area']:.2f}mm2  n={r['n_frames']}")

    # Helper: get position at a specific frame
    def get_pos_at_frame(cid, frame):
        data = working_df[
            (working_df['canonical_track_id'] == cid) &
            (working_df['frame'] <= frame)
        ].sort_values('frame')
        if len(data) == 0:
            data = working_df[working_df['canonical_track_id'] == cid].sort_values('frame')
        if len(data) == 0:
            return None
        return np.array([data['centroid_x'].iloc[-1], data['centroid_y'].iloc[-1]])

    # --- Collect ALL candidate pairs ---
    all_candidates = []
    for i, new_row in can_info_df.iterrows():
        new_cid = int(new_row['cid'])
        new_start = int(new_row['start_frame'])
        new_pos = np.array([new_row['start_x'], new_row['start_y']])
        new_area = new_row['mean_area']

        for j, pred_row in can_info_df.iterrows():
            pred_cid = int(pred_row['cid'])
            if pred_cid == new_cid:
                continue
            if pred_row['start_frame'] >= new_start:
                continue
            pred_end = int(pred_row['end_frame'])

            # GAP case: predecessor ended before successor started
            if pred_end < new_start:
                if pred_end < new_start - LOCAL_MAX_GAP_FRAMES:
                    continue  # gap too large
                pred_pos = np.array([pred_row['end_x'], pred_row['end_y']])
                dist = np.linalg.norm(new_pos - pred_pos)

            # OVERLAP case: predecessor still alive when successor starts
            else:
                overlap_frames = pred_end - new_start
                if overlap_frames > HANDOFF_OVERLAP_LIMIT:
                    continue  # predecessor keeps going = real budding event
                # Compare positions at the SAME frame during overlap
                compare_frame = new_start + min(5, overlap_frames)
                pred_pos = get_pos_at_frame(pred_cid, compare_frame)
                new_pos_ov = get_pos_at_frame(new_cid, compare_frame)
                if pred_pos is None or new_pos_ov is None:
                    continue
                dist = np.linalg.norm(pred_pos - new_pos_ov)

            if dist > LOCAL_MAX_STITCH_DIST:
                continue

            area_diff = abs(new_area - pred_row['mean_area'])
            score = dist + area_diff * 100  # very heavy area weight
            all_candidates.append((score, new_cid, pred_cid, dist, area_diff))

    # --- Greedy 1:1 assignment ---
    all_candidates.sort()
    local_map = {int(r['cid']): int(r['cid']) for _, r in can_info_df.iterrows()}

    def find_root(cid, mapping):
        visited = set()
        while mapping[cid] != cid:
            if cid in visited:
                break
            visited.add(cid)
            cid = mapping[cid]
        return cid

    used_as_successor = set()
    extended_tails = set()

    for score, new_cid, pred_cid, dist, area_diff in all_candidates:
        if new_cid in used_as_successor:
            continue
        root_new = find_root(new_cid, local_map)
        root_pred = find_root(pred_cid, local_map)
        if root_new == root_pred:
            continue
        if pred_cid in extended_tails:
            continue

        local_map[root_new] = root_pred
        used_as_successor.add(new_cid)
        extended_tails.add(pred_cid)
        print(f"  Stitch: cid {new_cid} -> continues cid {pred_cid} "
              f"(root {root_pred}, dist={dist:.0f}px, area_diff={area_diff:.2f}mm2, score={score:.1f})")

    # Flatten
    super_map = {cid: find_root(cid, local_map) for cid in local_map}

    # --- SECOND PASS: rescue orphans ---
    chains = {}
    for cid, root in super_map.items():
        chains.setdefault(root, []).append(cid)
    orphans = [root for root, members in chains.items() if len(members) == 1]
    print(f"\n  Orphaned fragments after pass 1: {len(orphans)}")

    for orphan_cid in orphans:
        orph = can_info_df[can_info_df['cid'] == orphan_cid]
        if len(orph) == 0:
            continue
        orph = orph.iloc[0]
        orph_end_pos = np.array([orph['end_x'], orph['end_y']])
        orph_start_pos = np.array([orph['start_x'], orph['start_y']])
        orph_area = orph['mean_area']

        best_chain_root = None
        best_score = float('inf')
        best_mode = None

        for chain_root, chain_cids in chains.items():
            if chain_root == orphan_cid or len(chain_cids) < 2:
                continue
            chain_data = can_info_df[can_info_df['cid'].isin(chain_cids)].sort_values('start_frame')

            # PREPEND
            earliest = chain_data.iloc[0]
            gap = earliest['start_frame'] - orph['end_frame']
            if -HANDOFF_OVERLAP_LIMIT < gap < LOCAL_MAX_GAP_FRAMES:
                dist = np.linalg.norm(orph_end_pos - np.array([earliest['start_x'], earliest['start_y']]))
                area_diff = abs(orph_area - earliest['mean_area'])
                score = dist + area_diff * 100
                if score < best_score and dist < LOCAL_MAX_STITCH_DIST * 2:
                    best_score, best_chain_root, best_mode = score, chain_root, 'prepend'

            # APPEND
            latest = chain_data.iloc[-1]
            gap = orph['start_frame'] - latest['end_frame']
            if -HANDOFF_OVERLAP_LIMIT < gap < LOCAL_MAX_GAP_FRAMES:
                dist = np.linalg.norm(orph_start_pos - np.array([latest['end_x'], latest['end_y']]))
                area_diff = abs(orph_area - latest['mean_area'])
                score = dist + area_diff * 100
                if score < best_score and dist < LOCAL_MAX_STITCH_DIST * 2:
                    best_score, best_chain_root, best_mode = score, chain_root, 'append'

        if best_chain_root is not None:
            if best_mode == 'prepend':
                local_map[best_chain_root] = orphan_cid
            else:
                local_map[orphan_cid] = best_chain_root
            print(f"  Orphan rescue: cid {orphan_cid} {best_mode}ed to chain {best_chain_root} (score={best_score:.1f})")

    super_map = {cid: find_root(cid, local_map) for cid in local_map}
    chains = {}
    for cid, root in super_map.items():
        chains.setdefault(root, []).append(cid)
    print(f"  Orphans remaining: {len([r for r,m in chains.items() if len(m)==1])}")

    # Apply to dataframes
    working_df['super_canonical'] = working_df['canonical_track_id'].map(super_map)
    extracted_df['super_canonical'] = extracted_df['canonical_track_id'].astype(float).map(
        lambda x: super_map.get(int(x), int(x)) if pd.notna(x) else np.nan
    )

    n_before = working_df['canonical_track_id'].nunique()
    n_after = working_df['super_canonical'].nunique()
    print(f"\nRe-stitching result: {n_before} fragments -> {n_after} physical plants")

    # Print chain details
    for root, cids in sorted(chains.items(), key=lambda x: len(x[1]), reverse=True):
        cid_list = sorted(cids)
        frames = can_info_df[can_info_df['cid'].isin(cids)]
        h_s = frames['start_frame'].min() * MINUTES_PER_FRAME / 60
        h_e = frames['end_frame'].max() * MINUTES_PER_FRAME / 60
        print(f"  Plant {root}: fragments {cid_list}, span {h_s:.1f}-{h_e:.1f}h")

    # ==========================================================
    # DETECT DIVISION (BUDDING) EVENTS
    # ==========================================================
    super_summaries = []
    for sid, group in working_df.groupby('super_canonical'):
        sg = group.sort_values('frame')
        super_summaries.append({
            'sid': int(sid),
            'start_frame': int(sg['frame'].iloc[0]),
            'end_frame': int(sg['frame'].iloc[-1]),
            'start_x': float(sg['centroid_x'].iloc[0]),
            'start_y': float(sg['centroid_y'].iloc[0]),
            'start_hours': float(sg['hours'].iloc[0]),
        })
    super_info_df = pd.DataFrame(super_summaries).sort_values('start_frame').reset_index(drop=True)

    divisions = []
    for _, child_row in super_info_df.iterrows():
        child_sid = int(child_row['sid'])
        child_start = child_row['start_frame']
        child_pos = np.array([child_row['start_x'], child_row['start_y']])
        child_hours = child_row['start_hours']

        # Find super-canonical plants alive when child appeared
        alive = super_info_df[
            (super_info_df['start_frame'] < child_start) &
            (super_info_df['end_frame'] >= child_start - 10) &
            (super_info_df['sid'] != child_sid)
        ]
        if len(alive) == 0:
            continue

        # Closest alive parent
        best_parent = None
        best_dist = float('inf')
        for _, par_row in alive.iterrows():
            par_data = working_df[
                (working_df['super_canonical'] == par_row['sid']) &
                (working_df['frame'] <= child_start)
            ].sort_values('frame')
            if len(par_data) == 0:
                continue
            par_pos = np.array([par_data['centroid_x'].iloc[-1],
                                par_data['centroid_y'].iloc[-1]])
            d = np.linalg.norm(child_pos - par_pos)
            if d < best_dist and d < 100:
                best_dist = d
                best_parent = int(par_row['sid'])

        if best_parent is not None:
            divisions.append({
                'parent': best_parent, 'child': child_sid,
                'hours': child_hours, 'dist': best_dist,
            })

    print(f"\nBudding events: {len(divisions)}")
    for d in divisions:
        print(f"  Plant {d['parent']} budded -> Plant {d['child']} at t={d['hours']:.1f}h")

    # ==========================================================
    # PLOTTING
    # ==========================================================
    cluster_id = working_df['cluster'].dropna().iloc[0]
    all_frames = sorted(working_df['frame'].unique())
    n_images = 6
    frame_indices = np.linspace(0, len(all_frames) - 1, n_images, dtype=int)
    selected_frames = [all_frames[i] for i in frame_indices]

    ref_img_rgb = get_reference_frame(0)
    h, w = ref_img_rgb.shape[:2]
    if DISH_ROI is not None:
        x_min, x_max = DISH_ROI['x_min'], DISH_ROI['x_max']
        y_min, y_max = DISH_ROI['y_min'], DISH_ROI['y_max']
    else:
        pad = 100
        x_min = max(0, int(working_df['centroid_x'].min() - pad))
        x_max = min(w, int(working_df['centroid_x'].max() + pad))
        y_min = max(0, int(working_df['centroid_y'].min() - pad))
        y_max = min(h, int(working_df['centroid_y'].max() + pad))

    # Filter: keep super-canonical plants with enough data
    MIN_TRACK_LENGTH = 10
    selected_frames_set = set(selected_frames)
    plotted_super_set = set()
    for sid in working_df['super_canonical'].dropna().unique():
        sdata = working_df[working_df['super_canonical'] == sid]
        if len(sdata) >= MIN_TRACK_LENGTH:
            if set(sdata['frame'].unique()).intersection(selected_frames_set):
                plotted_super_set.add(int(sid))

    # Assign ONE color per super-canonical plant
    distinct_colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e67e22', '#7f7f7f', '#bcbd22', '#17becf',
        '#c0392b', '#4ecdc4', '#45b7d1', '#96ceb4',
    ]
    super_color_map = {}
    for i, sid in enumerate(sorted(plotted_super_set)):
        super_color_map[sid] = distinct_colors[i % len(distinct_colors)]

    fig = plt.figure(figsize=(16, 11))
    gs = gridspec.GridSpec(2, n_images, height_ratios=[1, 2],
                           hspace=0.08, wspace=0.3, top=0.88)

    # ---- TOP ROW: Crop images ----
    for idx, frame_num in enumerate(selected_frames):
        ax_img = fig.add_subplot(gs[0, idx])
        frame_data = extracted_df[extracted_df['frame'] == frame_num].copy()
        hours = frame_num * MINUTES_PER_FRAME / 60
        plant_count = 0

        if len(frame_data) > 0:
            if USE_VIDEO:
                img_bgr = get_reference_frame(frame_num)
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            else:
                img_file = frame_data['image_file'].iloc[0]
                img_path = Path(IMAGE_DIRECTORY) / img_file
                img = cv2.imread(str(img_path))
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            cropped = img_rgb[y_min:y_max, x_min:x_max]
            gray = cv2.cvtColor(cropped, cv2.COLOR_RGB2GRAY)
            seg_frame = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB).astype(np.float32)

            for _, row in frame_data.iterrows():
                sc = row.get('super_canonical')
                if pd.isna(sc):
                    continue
                sc = int(sc)
                if sc not in plotted_super_set:
                    continue
                plant_count += 1

                hex_col = super_color_map[sc].lstrip('#')
                mask_rgb = np.array([int(hex_col[i:i+2], 16) for i in (0, 2, 4)])
                mask_color = mask_rgb.astype(np.float32)
                border_color = tuple(mask_rgb.astype(int).tolist())

                x1 = int(row['bbox_minc'] - x_min)
                y1 = int(row['bbox_minr'] - y_min)
                x2 = int(row['bbox_maxc'] - x_min)
                y2 = int(row['bbox_maxr'] - y_min)
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                bw, bh = x2 - x1, y2 - y1

                x1 = max(0, min(x1, seg_frame.shape[1] - 1))
                y1 = max(0, min(y1, seg_frame.shape[0] - 1))
                x2 = max(0, min(x2, seg_frame.shape[1] - 1))
                y2 = max(0, min(y2, seg_frame.shape[0] - 1))
                cx = max(0, min(cx, seg_frame.shape[1] - 1))
                cy = max(0, min(cy, seg_frame.shape[0] - 1))

                if x2 > x1 and y2 > y1 and bw > 0 and bh > 0:
                    mask = np.zeros((seg_frame.shape[0], seg_frame.shape[1]), dtype=np.uint8)
                    axes_len = (max(1, bw // 2), max(1, bh // 2))
                    cv2.ellipse(mask, (cx, cy), axes_len, 0, 0, 360, 255, -1)
                    alpha = 0.6
                    mb = (mask > 0).astype(np.float32)[:, :, np.newaxis]
                    for c in range(3):
                        seg_frame[:, :, c] = (seg_frame[:, :, c] * (1 - alpha * mb[:, :, 0])
                                              + mask_color[c] * alpha * mb[:, :, 0])
                    seg_frame_uint = seg_frame.astype(np.uint8)
                    cv2.ellipse(seg_frame_uint, (cx, cy), axes_len, 0, 0, 360, border_color, 2)

                    label = str(sc)
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    fs, th = 0.4, 1
                    (tw, tht), _ = cv2.getTextSize(label, font, fs, th)
                    lx, ly = cx - tw // 2, cy + tht // 2
                    cv2.rectangle(seg_frame_uint, (lx-2, ly-tht-2), (lx+tw+2, ly+2),
                                  tuple(mask_rgb.astype(int).tolist()), -1)
                    cv2.rectangle(seg_frame_uint, (lx-2, ly-tht-2), (lx+tw+2, ly+2), border_color, 1)
                    cv2.putText(seg_frame_uint, label, (lx, ly), font, fs, (255, 255, 0), th)
                    seg_frame = seg_frame_uint.astype(np.float32)

            ax_img.imshow(seg_frame.astype(np.uint8))
            plant_word = 'plant' if plant_count == 1 else 'plants'
            ax_img.set_title(f't = {hours:.1f}h\n{plant_count} {plant_word}',
                             fontsize=VIZ_AXIS_FONTSIZE, fontweight='bold')
            ax_img.axis('off')

    # ---- BOTTOM: Area plot ----
    ax_area = fig.add_subplot(gs[1, :])
    plotted_ids_ordered = []

    for sid in sorted(plotted_super_set):
        sdata = working_df[working_df['super_canonical'] == sid].sort_values('frame')
        if len(sdata) == 0:
            continue
        plotted_ids_ordered.append(sid)

        window_hours = 5
        wf = int(window_hours * 60 / MINUTES_PER_FRAME)
        sdata_tmp = sdata.copy()
        sdata_tmp['area_sm'] = sdata_tmp['area_mm2'].rolling(
            window=wf, center=True, min_periods=1).mean()
        areas = np.maximum(sdata_tmp['area_sm'].values, 0.001)

        ax_area.plot(sdata['hours'].values, areas, linewidth=2.5, alpha=0.85,
                     color=super_color_map[sid], label=f'Plant {sid}')

    # Budding markers on parent curve
    for d in divisions:
        p_sid = d['parent']
        c_sid = d['child']
        div_h = d['hours']
        if p_sid not in plotted_super_set:
            continue
        # Find parent area at budding time
        p_data = working_df[working_df['super_canonical'] == p_sid].sort_values('frame')
        nearest = p_data.iloc[(p_data['hours'] - div_h).abs().argsort().iloc[0]]
        p_area = nearest['area_mm2']
        p_color = super_color_map.get(p_sid, 'white')
        c_color = super_color_map.get(c_sid, 'gray')

        # Triangle marker on parent curve
        ax_area.plot(div_h, p_area, marker='v', markersize=8,
                     color=c_color, markeredgecolor='black', markeredgewidth=1,
                     zorder=10)
        # Small label next to marker (no arrow)
        ax_area.text(div_h + 1.5, p_area, f'budded {c_sid}',
                     fontsize=8, fontweight='bold', color=c_color,
                     va='center', ha='left', zorder=10)

    # Red dashed lines for timepoints
    for frame_num in selected_frames:
        fdata = working_df[working_df['frame'] == frame_num]
        if len(fdata) > 0:
            ax_area.axvline(fdata['hours'].iloc[0], color='red',
                            linestyle='--', linewidth=2, alpha=0.7)

    # Day/night shading
    max_hours = working_df['hours'].max()
    for ps in np.arange(0, max_hours + 24, 24):
        if ps <= max_hours:
            ne = min(ps + 12, max_hours)
            ax_area.axvspan(ps, ne, alpha=0.15, color='gray', zorder=0)

    ax_area.set_xlabel('Time (hours)', fontsize=28, fontweight='bold')
    ax_area.set_ylabel('Area (mm²)', fontsize=28, fontweight='bold')
    ax_area.set_ylim(bottom=0)
    ax_area.grid(False)
    ax_area.spines['top'].set_visible(False)
    ax_area.spines['right'].set_visible(False)
    ax_area.tick_params(labelsize=20)
    for lbl in ax_area.get_xticklabels() + ax_area.get_yticklabels():
        lbl.set_fontweight('bold')

    # ---- LEGEND ----
    from matplotlib.lines import Line2D
    from matplotlib.legend_handler import HandlerBase
    from matplotlib.patches import Patch

    class StackedLinesHandler(HandlerBase):
        def __init__(self, colors, **kwargs):
            self.colors = colors
            super().__init__(**kwargs)
        def create_artists(self, legend, orig_handle, xdescent, ydescent,
                           width, height, fontsize, trans):
            lines = []
            n = len(self.colors)
            sp = 4
            sy = ydescent + (height - n * sp) / 2 + sp / 2
            for i, col in enumerate(self.colors):
                l = Line2D([xdescent, xdescent + width], [sy + i*sp, sy + i*sp],
                           color=col, linewidth=2, alpha=0.85)
                l.set_transform(trans)
                lines.append(l)
            return lines

    track_colors = [super_color_map[s] for s in plotted_ids_ordered[:5]]
    stacked_h = Line2D([], [])
    red_l = Line2D([0], [0], color='red', linestyle='--', linewidth=2)
    night_p = Patch(facecolor='gray', alpha=0.15, edgecolor='none')
    day_p = Patch(facecolor='white', edgecolor='gray', linewidth=0.5)
    div_marker = Line2D([0], [0], marker='v', color='gray', markersize=6,
                        markeredgecolor='black', linestyle='None')

    fig.legend([stacked_h, div_marker, red_l, night_p, day_p],
               ['Tracks', 'Budding', 'Image timepoints', 'Night', 'Day'],
               loc='upper center', bbox_to_anchor=(0.5, 0.95), ncol=5,
               fontsize=16, frameon=True, fancybox=True, shadow=False,
               edgecolor='gray',
               handler_map={stacked_h: StackedLinesHandler(track_colors)})

    filter_label = " (FILTERED)" if use_filtered else " (RAW)"
    # plt.suptitle(f'Cluster {int(cluster_id)}: Individual Cell Growth — Canonical IDs{filter_label}',
    #              fontsize=VIZ_TITLE_FONTSIZE + 4, fontweight='bold', y=0.99)

    n_raw = working_df['track_id'].nunique()
    print(f"\nSummary: {n_raw} raw track IDs -> {len(plotted_super_set)} physical plants plotted")
    for sid in sorted(plotted_super_set):
        fragments = working_df[working_df['super_canonical'] == sid]['canonical_track_id'].unique()
        raw_ids = working_df[working_df['super_canonical'] == sid]['track_id'].unique()
        print(f"  Plant {sid}: raw IDs {sorted(raw_ids.astype(int))} "
              f"(canonical fragments: {sorted(fragments.astype(int))})")

    fig_path = OUTPUT_DIR + f"cluster_{int(cluster_id)}_area_with_crops_canonical.png"
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved: {fig_path}")
    plt.show()

    # Reset font settings
    plt.rcParams['font.size'] = 10
    plt.rcParams['font.weight'] = 'normal'
    plt.rcParams['axes.labelsize'] = 10
    plt.rcParams['axes.labelweight'] = 'normal'
    plt.rcParams['axes.titlesize'] = 10
    plt.rcParams['axes.titleweight'] = 'normal'
    plt.rcParams['xtick.labelsize'] = 10
    plt.rcParams['ytick.labelsize'] = 10
else:
    print("No extracted cluster data found.")

In [ ]:
# ============================================
# DIAGNOSE & FIX REMAINING TRACK FRAGMENTATIONS
# ============================================
# Reviewer #7: A plant's trajectory changes color mid-way, suggesting
# the same plant was tracked under two different IDs.
#
# This cell:
#   1. Scans for suspicious track transitions (one ends, another starts
#      at nearly the same position within a short gap) that were NOT
#      stitched by the super_canonical logic above
#   2. Reports them for inspection
#   3. Optionally force-merges them

import warnings
warnings.filterwarnings('ignore')

if 'super_canonical' in working_df.columns and len(working_df) > 0:
    print("=" * 60)
    print("TRACK FRAGMENTATION DIAGNOSTIC")
    print("=" * 60)

    # Build per-super_canonical summary
    sc_summaries = []
    for sid, group in working_df.groupby('super_canonical'):
        sg = group.sort_values('frame')
        sc_summaries.append({
            'sid': int(sid),
            'start_frame': int(sg['frame'].iloc[0]),
            'end_frame': int(sg['frame'].iloc[-1]),
            'start_x': float(sg['centroid_x'].iloc[0]),
            'start_y': float(sg['centroid_y'].iloc[0]),
            'end_x': float(sg['centroid_x'].iloc[-1]),
            'end_y': float(sg['centroid_y'].iloc[-1]),
            'mean_area': float(sg['area_mm2'].mean()),
            'n_frames': len(sg),
            'lifespan_h': (sg['frame'].iloc[-1] - sg['frame'].iloc[0]) * MINUTES_PER_FRAME / 60,
        })
    sc_df = pd.DataFrame(sc_summaries).sort_values('start_frame').reset_index(drop=True)

    # Find suspicious transitions: track A ends, track B starts nearby
    DIAG_MAX_GAP_H = 15      # hours
    DIAG_MAX_DIST_PX = 150   # pixels
    DIAG_MAX_AREA_RATIO = 3  # area must be within 3x

    gap_frames = DIAG_MAX_GAP_H * 60 / MINUTES_PER_FRAME
    suspicious = []

    for i, row_a in sc_df.iterrows():
        for j, row_b in sc_df.iterrows():
            if row_a['sid'] == row_b['sid']:
                continue
            # B starts after A ends (or small overlap)
            gap = row_b['start_frame'] - row_a['end_frame']
            if gap < -10 or gap > gap_frames:
                continue
            # Spatial proximity
            dist = np.sqrt((row_a['end_x'] - row_b['start_x'])**2 +
                          (row_a['end_y'] - row_b['start_y'])**2)
            if dist > DIAG_MAX_DIST_PX:
                continue
            # Area similarity
            if row_a['mean_area'] > 0 and row_b['mean_area'] > 0:
                ratio = max(row_a['mean_area'], row_b['mean_area']) / min(row_a['mean_area'], row_b['mean_area'])
                if ratio > DIAG_MAX_AREA_RATIO:
                    continue
            else:
                ratio = float('inf')

            gap_h = gap * MINUTES_PER_FRAME / 60
            suspicious.append({
                'track_A': int(row_a['sid']),
                'track_B': int(row_b['sid']),
                'A_end_h': row_a['end_frame'] * MINUTES_PER_FRAME / 60,
                'B_start_h': row_b['start_frame'] * MINUTES_PER_FRAME / 60,
                'gap_h': gap_h,
                'dist_px': dist,
                'area_A': row_a['mean_area'],
                'area_B': row_b['mean_area'],
                'area_ratio': ratio,
            })

    if suspicious:
        susp_df = pd.DataFrame(suspicious).sort_values('dist_px')
        # Remove duplicates (A->B and B->A)
        susp_df['pair'] = susp_df.apply(lambda r: tuple(sorted([r['track_A'], r['track_B']])), axis=1)
        susp_df = susp_df.drop_duplicates(subset='pair').drop(columns='pair')

        print(f"\nFound {len(susp_df)} suspicious track transitions:")
        print(f"(Criteria: gap < {DIAG_MAX_GAP_H}h, distance < {DIAG_MAX_DIST_PX}px, area ratio < {DIAG_MAX_AREA_RATIO}x)")
        print()
        for _, row in susp_df.iterrows():
            flag = " *** LIKELY SAME PLANT ***" if row['dist_px'] < 50 and row['area_ratio'] < 2 else ""
            print(f"  Plant {int(row['track_A'])} (ends {row['A_end_h']:.1f}h) -> "
                  f"Plant {int(row['track_B'])} (starts {row['B_start_h']:.1f}h)")
            print(f"    Gap: {row['gap_h']:.1f}h | Distance: {row['dist_px']:.0f}px | "
                  f"Area: {row['area_A']:.3f} -> {row['area_B']:.3f} mm² (ratio {row['area_ratio']:.1f}x){flag}")
            print()

        # ============================================
        # AUTO-MERGE: Force-stitch high-confidence matches
        # ============================================
        MERGE_MAX_DIST = 60       # pixels - very close
        MERGE_MAX_AREA_RATIO = 2  # areas must be similar
        MERGE_MAX_GAP_H = 10     # hours

        to_merge = susp_df[
            (susp_df['dist_px'] < MERGE_MAX_DIST) &
            (susp_df['area_ratio'] < MERGE_MAX_AREA_RATIO) &
            (susp_df['gap_h'].abs() < MERGE_MAX_GAP_H)
        ]

        if len(to_merge) > 0:
            print(f"\n{'='*60}")
            print(f"AUTO-MERGING {len(to_merge)} high-confidence matches")
            print(f"(dist < {MERGE_MAX_DIST}px, area ratio < {MERGE_MAX_AREA_RATIO}x, gap < {MERGE_MAX_GAP_H}h)")
            print(f"{'='*60}")

            merge_map = {}
            for _, row in to_merge.iterrows():
                a, b = int(row['track_A']), int(row['track_B'])
                # Keep the earlier track as the canonical
                if row['A_end_h'] < row['B_start_h']:
                    merge_map[b] = a  # B continues A
                else:
                    merge_map[a] = b  # A continues B

            # Resolve chains (A->B->C becomes A->A, B->A, C->A)
            def resolve(tid, mapping):
                visited = set()
                while tid in mapping and tid not in visited:
                    visited.add(tid)
                    tid = mapping[tid]
                return tid

            final_merge = {k: resolve(v, merge_map) for k, v in merge_map.items()}

            # Apply to working_df and extracted_df
            n_before = working_df['super_canonical'].nunique()
            working_df['super_canonical'] = working_df['super_canonical'].map(
                lambda x: final_merge.get(int(x), int(x))).astype(int)
            extracted_df['super_canonical'] = extracted_df['super_canonical'].map(
                lambda x: final_merge.get(int(x), int(x)) if pd.notna(x) else x)
            n_after = working_df['super_canonical'].nunique()

            for k, v in final_merge.items():
                print(f"  Merged plant {k} into plant {v}")
            print(f"\nPlants: {n_before} -> {n_after}")

            # Update color map
            super_ids = sorted(working_df['super_canonical'].dropna().unique())
            bright_colors_list = [
                '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
                '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7',
            ]
            super_color_map = {sid: bright_colors_list[i % len(bright_colors_list)] for i, sid in enumerate(super_ids)}
            print("\nColor map updated. Re-run the area/lineage plots to see corrected colors.")
        else:
            print("\nNo high-confidence auto-merges found.")
            print("Review the suspicious transitions above manually.")
    else:
        print("\nNo suspicious track transitions found — stitching looks clean!")
else:
    print("Run cell 48 (area with canonical IDs) first to build super_canonical.")


## 5.3c Lineage Timeline with Crop Images — Canonical IDs
Same crop-image-on-top layout as 5.3b, but the bottom panel shows a
**lineage timeline** (horizontal lifetime bars + budding connectors)
instead of the area plot. Uses the same `super_canonical` IDs and colors.

In [ ]:
# ============================================
# LINEAGE TIMELINE WITH CROP IMAGES - CANONICAL IDs
# ============================================
# Requires: super_canonical, super_color_map, divisions
#           (all produced by the 5.3b cell above)

if 'super_canonical' not in working_df.columns:
    print("Run 5.3b first to build super_canonical IDs.")
elif len(extracted_df) == 0:
    print("No extracted cluster data found.")
else:
    # --- Font setup (publication-ready) ---
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = 20
    plt.rcParams['font.weight'] = 'bold'
    plt.rcParams['axes.labelsize'] = 24
    plt.rcParams['axes.labelweight'] = 'bold'
    plt.rcParams['axes.titlesize'] = 26
    plt.rcParams['axes.titleweight'] = 'bold'
    plt.rcParams['xtick.labelsize'] = 20
    plt.rcParams['ytick.labelsize'] = 20
    plt.rcParams['legend.fontsize'] = 18

    cluster_id = working_df['cluster'].dropna().iloc[0]

    # --- Frame selection (same 6 as area plot) ---
    all_frames = sorted(working_df['frame'].unique())
    n_images = 6
    frame_indices = np.linspace(0, len(all_frames) - 1, n_images, dtype=int)
    selected_frames = [all_frames[i] for i in frame_indices]

    # --- ROI ---
    ref_img_rgb = get_reference_frame(0)
    h, w = ref_img_rgb.shape[:2]
    if DISH_ROI is not None:
        x_min, x_max = DISH_ROI['x_min'], DISH_ROI['x_max']
        y_min, y_max = DISH_ROI['y_min'], DISH_ROI['y_max']
    else:
        pad = 100
        x_min = max(0, int(working_df['centroid_x'].min() - pad))
        x_max = min(w, int(working_df['centroid_x'].max() + pad))
        y_min = max(0, int(working_df['centroid_y'].min() - pad))
        y_max = min(h, int(working_df['centroid_y'].max() + pad))

    # --- Build lifetime info per super-canonical plant ---
    life = {}  # sid -> (start_h, end_h)
    for sid in sorted(plotted_super_set):
        sdata = working_df[working_df['super_canonical'] == sid].sort_values('frame')
        if len(sdata) == 0:
            continue
        life[sid] = (float(sdata['hours'].iloc[0]), float(sdata['hours'].iloc[-1]))

    if not life:
        print("No lifetime data.")
    else:
        # --- Row assignment: parents on top, children below ---
        # Build parent->children map from divisions
        children_map = {}
        parent_map = {}
        for d in divisions:
            p, c = d['parent'], d['child']
            if p in life and c in life:
                children_map.setdefault(p, []).append(c)
                parent_map[c] = p

        # Roots = plants with no parent
        roots = [sid for sid in life if sid not in parent_map]
        roots.sort(key=lambda s: life[s][0])  # earliest first

        row_for = {}
        _row = [0]  # mutable counter (nonlocal workaround for notebook scope)

        def assign_rows(sid, row_start):
            """Assign row to plant and its children recursively."""
            row_for[sid] = row_start
            _row[0] = row_start + 1
            for child in sorted(children_map.get(sid, []), key=lambda c: life[c][0]):
                if child not in row_for:
                    assign_rows(child, _row[0])

        for r in roots:
            assign_rows(r, _row[0])
        current_row = _row[0]

        # Any remaining plants not in the tree
        for sid in life:
            if sid not in row_for:
                row_for[sid] = current_row
                current_row += 1

        n_rows = current_row

        # --- Figure layout ---
        fig = plt.figure(figsize=(16, 11))
        gs = gridspec.GridSpec(2, n_images, height_ratios=[1, 2],
                               hspace=0.08, wspace=0.3, top=0.88)

        # =========================================================
        # TOP ROW: Crop images (identical to 5.3b)
        # =========================================================
        for idx, frame_num in enumerate(selected_frames):
            ax_img = fig.add_subplot(gs[0, idx])
            frame_data = extracted_df[extracted_df['frame'] == frame_num].copy()
            hours = frame_num * MINUTES_PER_FRAME / 60
            plant_count = 0

            if len(frame_data) > 0:
                if USE_VIDEO:
                    img_bgr = get_reference_frame(frame_num)
                    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                else:
                    img_file = frame_data['image_file'].iloc[0]
                    img_path = Path(IMAGE_DIRECTORY) / img_file
                    img = cv2.imread(str(img_path))
                    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                cropped = img_rgb[y_min:y_max, x_min:x_max]
                gray = cv2.cvtColor(cropped, cv2.COLOR_RGB2GRAY)
                seg_frame = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB).astype(np.float32)

                for _, row in frame_data.iterrows():
                    sc = row.get('super_canonical')
                    if pd.isna(sc):
                        continue
                    sc = int(sc)
                    if sc not in plotted_super_set:
                        continue
                    plant_count += 1

                    hex_col = super_color_map[sc].lstrip('#')
                    mask_rgb = np.array([int(hex_col[i:i+2], 16) for i in (0, 2, 4)])
                    mask_color = mask_rgb.astype(np.float32)
                    border_color = tuple(mask_rgb.astype(int).tolist())

                    x1 = int(row['bbox_minc'] - x_min)
                    y1 = int(row['bbox_minr'] - y_min)
                    x2 = int(row['bbox_maxc'] - x_min)
                    y2 = int(row['bbox_maxr'] - y_min)
                    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                    bw, bh = x2 - x1, y2 - y1

                    x1 = max(0, min(x1, seg_frame.shape[1] - 1))
                    y1 = max(0, min(y1, seg_frame.shape[0] - 1))
                    x2 = max(0, min(x2, seg_frame.shape[1] - 1))
                    y2 = max(0, min(y2, seg_frame.shape[0] - 1))
                    cx = max(0, min(cx, seg_frame.shape[1] - 1))
                    cy = max(0, min(cy, seg_frame.shape[0] - 1))

                    if x2 > x1 and y2 > y1 and bw > 0 and bh > 0:
                        mask = np.zeros((seg_frame.shape[0], seg_frame.shape[1]), dtype=np.uint8)
                        axes_len = (max(1, bw // 2), max(1, bh // 2))
                        cv2.ellipse(mask, (cx, cy), axes_len, 0, 0, 360, 255, -1)
                        alpha = 0.6
                        mb = (mask > 0).astype(np.float32)[:, :, np.newaxis]
                        for c in range(3):
                            seg_frame[:, :, c] = (seg_frame[:, :, c] * (1 - alpha * mb[:, :, 0])
                                                  + mask_color[c] * alpha * mb[:, :, 0])
                        seg_frame_uint = seg_frame.astype(np.uint8)
                        cv2.ellipse(seg_frame_uint, (cx, cy), axes_len, 0, 0, 360, border_color, 2)

                        label = str(sc)
                        font = cv2.FONT_HERSHEY_SIMPLEX
                        fs, th = 0.4, 1
                        (tw, tht), _ = cv2.getTextSize(label, font, fs, th)
                        lx, ly = cx - tw // 2, cy + tht // 2
                        cv2.rectangle(seg_frame_uint, (lx-2, ly-tht-2), (lx+tw+2, ly+2),
                                      tuple(mask_rgb.astype(int).tolist()), -1)
                        cv2.rectangle(seg_frame_uint, (lx-2, ly-tht-2), (lx+tw+2, ly+2), border_color, 1)
                        cv2.putText(seg_frame_uint, label, (lx, ly), font, fs, (255, 255, 0), th)
                        seg_frame = seg_frame_uint.astype(np.float32)

                ax_img.imshow(seg_frame.astype(np.uint8))
                plant_word = 'plant' if plant_count == 1 else 'plants'
                ax_img.set_title(f't = {hours:.1f}h\n{plant_count} {plant_word}',
                                 fontsize=VIZ_AXIS_FONTSIZE, fontweight='bold')
                ax_img.axis('off')

        # =========================================================
        # BOTTOM: Lineage timeline (L-connectors, white background)
        # =========================================================
        ax_lin = fig.add_subplot(gs[1, :])

        # Spacing between families
        FAMILY_GAP = 0.8
        LINE_SPACING = 1.0

        # Rebuild row assignments with family gaps
        row_for = {}
        y_pos = 0.0

        families_plotted = []
        for r in roots:
            family_members = [r]
            stack = list(sorted(children_map.get(r, []), key=lambda c: life[c][0]))
            while stack:
                child = stack.pop(0)
                if child in life and child not in family_members:
                    family_members.append(child)
                    # Add grandchildren
                    grandchildren = sorted(children_map.get(child, []), key=lambda c: life[c][0])
                    stack = grandchildren + stack
            families_plotted.append((r, family_members))

        for fam_idx, (root_sid, members) in enumerate(families_plotted):
            if fam_idx > 0:
                y_pos += FAMILY_GAP
            for member in members:
                row_for[member] = y_pos
                y_pos += LINE_SPACING

        for sid in life:
            if sid not in row_for:
                y_pos += FAMILY_GAP
                row_for[sid] = y_pos
                y_pos += LINE_SPACING

        # Horizontal lifetime lines (colored)
        for sid, (start_h, end_h) in life.items():
            y = row_for[sid]
            color = super_color_map.get(sid, '#808080')
            ax_lin.plot([start_h, end_h], [y, y], color=color,
                        linewidth=3, solid_capstyle='butt', zorder=3)

        # L-shaped budding connectors
        for d in divisions:
            p_sid, c_sid = d['parent'], d['child']
            div_h = d['hours']
            if p_sid not in row_for or c_sid not in row_for:
                continue
            y_p = row_for[p_sid]
            y_c = row_for[c_sid]
            c_color = super_color_map.get(c_sid, '#808080')

            # Vertical drop from parent to child
            ax_lin.plot([div_h, div_h], [y_p, y_c], color=c_color,
                        linewidth=1.5, zorder=2)

        # Red dashed lines for timepoints
        for frame_num in selected_frames:
            fdata = working_df[working_df['frame'] == frame_num]
            if len(fdata) > 0:
                ax_lin.axvline(fdata['hours'].iloc[0], color='red',
                               linestyle='--', linewidth=2, alpha=0.7, zorder=1)

        # Day/night shading
        max_hours = working_df['hours'].max()
        for ps in np.arange(0, max_hours + 24, 24):
            if ps <= max_hours:
                ne = min(ps + 12, max_hours)
                ax_lin.axvspan(ps, ne, alpha=0.15, color='gray', zorder=0)

        # Y-axis labels
        y_ticks = []
        y_labels = []
        inv_row = {v: k for k, v in row_for.items()}
        for y in sorted(inv_row.keys()):
            sid = inv_row[y]
            y_ticks.append(y)
            if sid in parent_map:
                y_labels.append(f'Daughter {sid}')
            else:
                y_labels.append(f'Mother {sid}')

        ax_lin.set_yticks(y_ticks)
        ax_lin.set_yticklabels(y_labels, fontsize=16, fontweight='bold')
        ax_lin.invert_yaxis()

        ax_lin.set_xlabel('Time (hours)', fontsize=28, fontweight='bold')
        ax_lin.set_xlim(left=-2, right=max_hours + 2)
        ax_lin.set_ylim(y_pos - 0.5, -0.5)
        ax_lin.grid(False)
        ax_lin.spines['top'].set_visible(False)
        ax_lin.spines['right'].set_visible(False)
        ax_lin.tick_params(labelsize=20)
        for lbl in ax_lin.get_xticklabels() + ax_lin.get_yticklabels():
            lbl.set_fontweight('bold')

        # ---- LEGEND ----
        from matplotlib.lines import Line2D
        from matplotlib.patches import Patch

        from matplotlib.legend_handler import HandlerBase
        class StackedLinesHandler(HandlerBase):
            def __init__(self, colors, **kwargs):
                self.colors = colors
                super().__init__(**kwargs)
            def create_artists(self, legend, orig_handle, xdescent, ydescent,
                               width, height, fontsize, trans):
                lines = []
                n = len(self.colors)
                sp = 4
                sy = ydescent + (height - n * sp) / 2 + sp / 2
                for i, col in enumerate(self.colors):
                    l = Line2D([xdescent, xdescent + width], [sy + i*sp, sy + i*sp],
                               color=col, linewidth=2, alpha=0.85)
                    l.set_transform(trans)
                    lines.append(l)
                return lines

        track_colors = [super_color_map[s] for s in sorted(plotted_super_set)[:5]]
        stacked_h = Line2D([], [])
        div_handle = Line2D([0], [0], color='gray', linewidth=1.5)
        red_l = Line2D([0], [0], color='red', linestyle='--', linewidth=2)
        night_p = Patch(facecolor='gray', alpha=0.15, edgecolor='none')
        day_p = Patch(facecolor='white', edgecolor='gray', linewidth=0.5)

        fig.legend([stacked_h, div_handle, red_l, night_p, day_p],
                   ['Tracks', 'Budding', 'Image timepoints', 'Night', 'Day'],
                   loc='upper center', bbox_to_anchor=(0.5, 0.95), ncol=5,
                   fontsize=16, frameon=True, fancybox=True, shadow=False,
                   edgecolor='gray',
                   handler_map={stacked_h: StackedLinesHandler(track_colors)})

        filter_label = " (FILTERED)" if use_filtered else " (RAW)"
        # plt.suptitle(f'Cluster {int(cluster_id)}: Lineage Timeline — Canonical IDs{filter_label}',
        #              fontsize=VIZ_TITLE_FONTSIZE + 4, fontweight='bold', y=0.99)

        fig_path = OUTPUT_DIR + f"cluster_{int(cluster_id)}_lineage_canonical.png"
        plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved: {fig_path}")
        plt.show()

        # Reset font settings
        plt.rcParams['font.size'] = 10
        plt.rcParams['font.weight'] = 'normal'
        plt.rcParams['axes.labelsize'] = 10
        plt.rcParams['axes.labelweight'] = 'normal'
        plt.rcParams['axes.titlesize'] = 10
        plt.rcParams['axes.titleweight'] = 'normal'
        plt.rcParams['xtick.labelsize'] = 10
        plt.rcParams['ytick.labelsize'] = 10

In [ ]:
# ============================================
# Cluster lineage timeline (Mother/Daughter style)
# Matches cleaned Cluster area plot (same tracks + time axis)
# ============================================

import numpy as np
from collections import defaultdict

if 'extracted_df' not in dir() or extracted_df is None or len(extracted_df) == 0:
    print("No extracted cluster data. Run cluster extraction first (Section 4.2).")
else:
    # 1) CLEAN WORKING DATA FOR THIS CLUSTER
    if 'extracted_df_filtered' in dir() and extracted_df_filtered is not None and len(extracted_df_filtered) > 0:
        cluster_df = extracted_df_filtered.copy()
        use_filtered = True
        print("✓ Using FILTERED extracted cluster data")
    else:
        cluster_df = extracted_df.copy()
        use_filtered = False
        print("⚠️ Using RAW extracted cluster data")

    # Ensure canonical IDs exist
    if 'canonical_track_id' not in cluster_df.columns:
        if 'track_mapping' in dir() and track_mapping is not None:
            cluster_df['canonical_track_id'] = cluster_df['track_id'].map(track_mapping)
        else:
            cluster_df['canonical_track_id'] = cluster_df['track_id']

    cluster_df = cluster_df.dropna(subset=['canonical_track_id']).copy()
    cluster_df['canonical_track_id'] = cluster_df['canonical_track_id'].astype(int)

    # 2) MATCH TIME AXIS + TRACKS TO AREA PLOT (same 6 timepoints, same length filter)
    all_frames = sorted(cluster_df['frame'].unique())
    n_images = 6
    frame_indices = np.linspace(0, len(all_frames) - 1, n_images, dtype=int)
    selected_frames = [all_frames[i] for i in frame_indices]

    MIN_TRACK_LENGTH = 10
    selected_frames_set = set(selected_frames)

    visible_tracks = []  # canonical IDs
    for cid, df_c in cluster_df.groupby('canonical_track_id'):
        if len(df_c) < MIN_TRACK_LENGTH:
            continue
        if set(df_c['frame'].unique()).intersection(selected_frames_set):
            visible_tracks.append(int(cid))

    visible_tracks = sorted(visible_tracks)
    print(f"Visible canonical tracks (match plot): {len(visible_tracks)}")

    if not visible_tracks:
        print("No tracks satisfy length + selected-frame criteria.")
    else:
        # 3) BUILD LINEAGE RELATIONSHIPS FOR THESE TRACKS
        if 'lineage_df' not in dir() or lineage_df is None or len(lineage_df) == 0:
            print("No lineage_df available. Run lineage inference (Section 3.4).")
        else:
            parent_col = 'parent_canonical' if 'parent_canonical' in lineage_df.columns else 'parent_id'
            child_col  = 'child_canonical'  if 'child_canonical'  in lineage_df.columns else 'child_id'

            ld = lineage_df.dropna(subset=[parent_col, child_col]).copy()
            ld[parent_col] = ld[parent_col].astype(int)
            ld[child_col]  = ld[child_col].astype(int)

            visible_set = set(visible_tracks)
            edges = []  # (parent, child)
            for _, row in ld.iterrows():
                p = int(row[parent_col])
                c = int(row[child_col])
                if p in visible_set and c in visible_set:
                    edges.append((p, c))

            parent_map = {c: p for p, c in edges}
            children_map = defaultdict(list)
            for p, c in edges:
                children_map[p].append(c)

            roots = [cid for cid in visible_tracks if cid not in parent_map]

            # 4) ROW ASSIGNMENT (Mother row, daughters underneath)
            row_for_track = {}
            current_row = 0

            def assign_family_rows(root_id, start_row):
                """Assign rows to a family starting at start_row; return next free row."""
                row = start_row
                stack = [root_id]
                while stack:
                    tid = stack.pop(0)
                    if tid not in row_for_track:
                        row_for_track[tid] = row
                        row += 1
                    for child in children_map.get(tid, []):
                        if child not in row_for_track:
                            row_for_track[child] = row
                            row += 1
                        stack.append(child)
                return row

            for r in roots:
                current_row = assign_family_rows(r, current_row)

            # Any remaining visible tracks with no lineage links
            for cid in visible_tracks:
                if cid not in row_for_track:
                    row_for_track[cid] = current_row
                    current_row += 1

            # 5) LIFETIMES FROM CLUSTER DATA
            life = {}  # cid -> (start_h, end_h)
            for cid in visible_tracks:
                df_c = cluster_df[cluster_df['canonical_track_id'] == cid].sort_values('frame')
                if len(df_c) == 0:
                    continue
                start_h = df_c['hours'].iloc[0]
                end_h   = df_c['hours'].iloc[-1]
                life[cid] = (start_h, end_h)

            if not life:
                print("No lifetime data for visible tracks.")
            else:
                # 6) HEURISTIC PARENT–CHILD EDGES FROM LIFETIMES (IGNORE lineage_df)
                ordered = sorted(life.keys(), key=lambda cid: life[cid][0])  # by birth time
                edges = []  # (parent, child)
                for i, child in enumerate(ordered):
                    child_start_h = life[child][0]
                    # earlier tracks alive when this child appears
                    candidates = [
                        parent for parent in ordered[:i]
                        if life[parent][0] <= child_start_h <= life[parent][1]
                    ]
                    if candidates:
                        # choose mother with latest end time
                        parent = max(candidates, key=lambda cid: life[cid][1])
                        edges.append((parent, child))

                # Budding time = child's birth time
                div_times = {(p, c): life[c][0] for (p, c) in edges}

                # 7) PLOT MOTHER/DAUGHTER TIMELINE
                fig, ax = plt.subplots(figsize=(12, 5))

                # Horizontal lifetimes
                for cid, (start_h, end_h) in life.items():
                    y = row_for_track[cid]
                    ax.hlines(y, start_h, end_h, color='white', linewidth=2)

                # Vertical budding connectors
                for (p, c), tdiv in div_times.items():
                    if p not in life or c not in life:
                        continue
                    y_p = row_for_track[p]
                    y_c = row_for_track[c]
                    ax.vlines(tdiv, y_p, y_c, color='white', linewidth=1.5)

                # Y labels
                inv_row = {v: k for k, v in row_for_track.items()}
                y_ticks = sorted(inv_row.keys())
                y_labels = [f"Track {inv_row[y]}" for y in y_ticks]
                ax.set_yticks(y_ticks)
                ax.set_yticklabels(y_labels)

                # Match time axis to lifetimes
                min_h = min(h[0] for h in life.values())
                max_h = max(h[1] for h in life.values())
                ax.set_xlim(min_h - 2, max_h + 2)
                ax.set_xlabel('Time (hours)')

                cluster_id = int(cluster_df['cluster'].dropna().iloc[0])
                ax.set_title(
                    f'Cluster {cluster_id}: Lineage timeline (heuristic)'
                    + (' (FILTERED)' if use_filtered else ' (RAW)')
                )

                # Red dashed lines at image timepoints (to match crop/area fig)
                for f in selected_frames:
                    h = f * MINUTES_PER_FRAME / 60.0
                    ax.axvline(h, color='red', linestyle='--', linewidth=1, alpha=0.5)

                # Styling (dark background similar to example)
                ax.set_facecolor('black')
                fig.patch.set_facecolor('black')
                ax.tick_params(colors='white')
                for spine in ax.spines.values():
                    spine.set_color('white')

                plt.tight_layout()
                plt.show()

## 5.4 Doubling Time Analysis
Calculate and visualize doubling time from budding events.

In [ ]:
# ============================================
# DOUBLING TIME: BUDDING vs NON-BUDDING PLANTS
# ============================================
# Reviewer concern: averaging doubling time across ALL plants (including
# those that never budded) is misleading. This cell separates the two
# groups and reports doubling time only from actively budding plants.
#
# Approach:
#   1. Identify which canonical track IDs appear as parents in lineage_df
#   2. Build the "budding lineage" = parents + all their descendants
#   3. "Non-budding" = everything outside that lineage
#   4. Build separate population curves for each group
#   5. Fit exponential growth and report both

from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

def exponential_growth(t, N0, r):
    return N0 * np.exp(r * t)

if len(extracted_df) > 0 and len(lineage_df) > 0:
    print("=" * 60)
    print("DOUBLING TIME: BUDDING LINEAGE vs NON-BUDDING ANALYSIS")
    print("=" * 60)

    # --- Step 1: Identify budding parents ---
    all_canonical = set(extracted_df['canonical_track_id'].dropna().unique())
    budding_parents = set(lineage_df['parent_canonical'].dropna().unique()) & all_canonical

    # --- Step 2: Build full budding lineage (parents + all descendants) ---
    # Start with parents, then walk the lineage tree to find all children,
    # grandchildren, etc.
    parent_to_children = {}
    for _, row in lineage_df.iterrows():
        p = row['parent_canonical']
        c = row['child_canonical']
        if pd.notna(p) and pd.notna(c):
            parent_to_children.setdefault(int(p), []).append(int(c))

    budding_lineage = set()
    queue = list(budding_parents)
    while queue:
        plant = queue.pop()
        if plant in budding_lineage:
            continue
        budding_lineage.add(plant)
        # Add all children of this plant
        for child in parent_to_children.get(int(plant), []):
            if child in all_canonical and child not in budding_lineage:
                queue.append(child)

    non_budding_plants = all_canonical - budding_lineage

    print(f"\nTotal unique plants (canonical): {len(all_canonical)}")
    print(f"  Budding lineage (parents + descendants): {len(budding_lineage)}")
    print(f"    - Parents (produced daughter): {len(budding_parents)}")
    print(f"    - Descendants (children/grandchildren): {len(budding_lineage - budding_parents)}")
    print(f"  Non-budding (never budded, no budding ancestor): {len(non_budding_plants)}")
    print(f"  Ratio in budding lineage: {len(budding_lineage)/len(all_canonical)*100:.1f}%")

    # --- Step 3: Lifespan stats per group ---
    track_lifespans = extracted_df.groupby('canonical_track_id').agg(
        first_frame=('frame', 'min'),
        last_frame=('frame', 'max'),
        n_frames=('frame', 'nunique')
    )
    track_lifespans['lifespan_hours'] = (track_lifespans['last_frame'] - track_lifespans['first_frame']) * MINUTES_PER_FRAME / 60

    budding_lifespans = track_lifespans.loc[track_lifespans.index.isin(budding_lineage)]
    nonbudding_lifespans = track_lifespans.loc[track_lifespans.index.isin(non_budding_plants)]

    print(f"\nLifespan (hours):")
    if len(budding_lifespans) > 0:
        print(f"  Budding lineage:    mean={budding_lifespans['lifespan_hours'].mean():.1f}, "
              f"median={budding_lifespans['lifespan_hours'].median():.1f}, "
              f"range=[{budding_lifespans['lifespan_hours'].min():.1f}, {budding_lifespans['lifespan_hours'].max():.1f}]")
    if len(nonbudding_lifespans) > 0:
        print(f"  Non-budding plants: mean={nonbudding_lifespans['lifespan_hours'].mean():.1f}, "
              f"median={nonbudding_lifespans['lifespan_hours'].median():.1f}, "
              f"range=[{nonbudding_lifespans['lifespan_hours'].min():.1f}, {nonbudding_lifespans['lifespan_hours'].max():.1f}]")

    # --- Step 4: Population curves per group ---
    frames = sorted(extracted_df['frame'].unique())
    hours = [f * MINUTES_PER_FRAME / 60 for f in frames]

    pop_all = []
    pop_lineage = []
    pop_nonbudding = []

    for f in frames:
        frame_data = extracted_df[extracted_df['frame'] == f]
        all_in_frame = set(frame_data['canonical_track_id'].dropna().unique())
        pop_all.append(len(all_in_frame))
        pop_lineage.append(len(all_in_frame & budding_lineage))
        pop_nonbudding.append(len(all_in_frame - budding_lineage))

    pop_df = pd.DataFrame({
        'frame': frames,
        'hours': hours,
        'all': pop_all,
        'budding_lineage': pop_lineage,
        'non_budding': pop_nonbudding
    })

    # --- Step 5: Fit exponential to each group ---
    results = {}
    for label, col in [('All plants', 'all'), ('Budding lineage', 'budding_lineage'), ('Non-budding', 'non_budding')]:
        y = np.array(pop_df[col].values, dtype=float)
        x = np.array(pop_df['hours'].values, dtype=float)
        if y[-1] > y[0] and y[0] > 0:
            try:
                popt, pcov = curve_fit(exponential_growth, x, y,
                                       p0=[y[0], 0.01], maxfev=10000)
                r = popt[1]
                if r > 0:
                    dt = np.log(2) / r
                    # R-squared
                    y_pred = exponential_growth(x, *popt)
                    ss_res = np.sum((y - y_pred) ** 2)
                    ss_tot = np.sum((y - np.mean(y)) ** 2)
                    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
                    results[label] = {'N0': popt[0], 'r': r, 'doubling_time': dt, 'R2': r_squared,
                                      'start_pop': int(y[0]), 'end_pop': int(y[-1])}
            except Exception as e:
                print(f"  Fit failed for {label}: {e}")
        elif y[0] == 0:
            print(f"  Skipped {label}: starts at 0")
        elif y[-1] <= y[0]:
            print(f"  Skipped {label}: no growth (pop {int(y[0])} -> {int(y[-1])})")

    print(f"\n{'='*60}")
    print("DOUBLING TIME RESULTS")
    print(f"{'='*60}")
    for label, res in results.items():
        print(f"\n  {label}:")
        print(f"    Population: {res['start_pop']} -> {res['end_pop']}")
        print(f"    Growth rate (r): {res['r']:.6f} /hour")
        print(f"    Doubling time: {res['doubling_time']:.1f} hours")
        print(f"    R\u00b2: {res['R2']:.4f}")

    # Also compute simple endpoint doubling time for comparison
    print(f"\n  --- Endpoint method (for reference) ---")
    for label, col in [('All plants', 'all'), ('Budding lineage', 'budding_lineage')]:
        y = pop_df[col].values
        if y[0] > 0 and y[-1] > y[0]:
            r_endpoint = np.log(y[-1] / y[0]) / (hours[-1] - hours[0])
            dt_endpoint = np.log(2) / r_endpoint
            print(f"  {label}: {int(y[0])} -> {int(y[-1])} in {hours[-1]-hours[0]:.0f}h = {dt_endpoint:.1f}h doubling time")

    # --- Step 6: Visualization ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: Population curves
    ax = axes[0]
    ax.step(pop_df['hours'], pop_df['all'], 'k-', linewidth=1.5, label='All plants', alpha=0.4, where='post')
    ax.step(pop_df['hours'], pop_df['budding_lineage'], '-', color='#228B22', linewidth=2.5,
            label='Budding lineage', where='post')
    ax.step(pop_df['hours'], pop_df['non_budding'], '--', color='#CC4444', linewidth=2,
            label='Non-budding', where='post')

    # Overlay exponential fits
    x_fit = np.linspace(pop_df['hours'].min(), pop_df['hours'].max(), 200)
    fit_styles = {'All plants': ('gray', ':'), 'Budding lineage': ('#228B22', '-.'), 'Non-budding': ('#CC4444', ':')}
    for label, res in results.items():
        color, ls = fit_styles.get(label, ('gray', ':'))
        y_fit = exponential_growth(x_fit, res['N0'], res['r'])
        ax.plot(x_fit, y_fit, ls, color=color, alpha=0.6, linewidth=1.5)

    ax.set_xlabel('Time (hours)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Plants', fontsize=14, fontweight='bold')
    ax.set_title('Population: Budding Lineage vs Non-Budding', fontsize=16, fontweight='bold')
    ax.legend(fontsize=11)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Right: Summary table
    ax2 = axes[1]
    ax2.axis('off')
    table_text = "DOUBLING TIME COMPARISON\n" + "=" * 40 + "\n\n"
    for label, res in results.items():
        table_text += f"{label}:\n"
        table_text += f"  Pop: {res['start_pop']} \u2192 {res['end_pop']}\n"
        table_text += f"  Doubling time: {res['doubling_time']:.1f} h  (curve fit)\n"
        table_text += f"  R\u00b2 = {res['R2']:.4f}\n\n"

    # Add endpoint values
    table_text += "--- Endpoint method ---\n"
    for label, col in [('All plants', 'all'), ('Budding lineage', 'budding_lineage')]:
        y = pop_df[col].values
        if y[0] > 0 and y[-1] > y[0]:
            r_ep = np.log(y[-1] / y[0]) / (hours[-1] - hours[0])
            dt_ep = np.log(2) / r_ep
            table_text += f"{label}: {dt_ep:.1f} h\n"

    table_text += f"\nPlants: {len(all_canonical)} total\n"
    table_text += f"Budding lineage: {len(budding_lineage)} ({len(budding_lineage)/len(all_canonical)*100:.0f}%)\n"
    table_text += f"Non-budding: {len(non_budding_plants)} ({len(non_budding_plants)/len(all_canonical)*100:.0f}%)"

    ax2.text(0.05, 0.95, table_text, transform=ax2.transAxes,
             fontsize=13, fontfamily='monospace', verticalalignment='top')

    plt.tight_layout()

    fig_path = OUTPUT_DIR + "doubling_time_budding_vs_nonbudding.png"
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved: {fig_path}")
    plt.show()

else:
    if len(extracted_df) == 0:
        print("No extracted cluster data. Run extraction first.")
    else:
        print("No lineage data. Run lineage detection first.")


## 5.5 Trajectories
Show movement paths of all tracked individuals.

In [ ]:
# ============================================
# TRACK TRAJECTORIES - STATIC IMAGE (LIKE REFERENCE)
# ============================================
# Shows all trajectories on full petri dish view
# Plants removed from background, only dots and trails visible

if len(unified_df) > 0:
    print("Creating trajectory visualization...")
    
    # Load reference image (full view, no crop)
    # Load reference image - works for both video and images
    ref_img_rgb = get_reference_frame(0)
    
    # =========================================================
    # REMOVE PLANTS: Use segmentation to mask out plant areas
    # =========================================================
    # Create a clean background by inpainting over plant locations
    clean_background = ref_img_rgb.copy()
    
    # Get all plant locations from tracking data
    all_centroids = unified_df[['centroid_x', 'centroid_y', 'frame']].dropna()
    
    # Create mask of plant regions (approximate with circles around centroids)
    plant_mask = np.zeros(ref_img_rgb.shape[:2], dtype=np.uint8)
    
    # For each frame's detections, mark plant areas
    for frame_num in unified_df['frame'].unique():
        frame_data = unified_df[unified_df['frame'] == frame_num]
        for _, row in frame_data.iterrows():
            if pd.notna(row['centroid_x']) and pd.notna(row['centroid_y']):
                cx, cy = int(row['centroid_x']), int(row['centroid_y'])
                # Use bbox if available, otherwise estimate size
                if 'bbox_minr' in row and pd.notna(row['bbox_minr']):
                    h = int(row['bbox_maxr'] - row['bbox_minr'])
                    w = int(row['bbox_maxc'] - row['bbox_minc'])
                    radius = max(h, w) // 2 + 5
                else:
                    radius = 30  # Default radius
                cv2.circle(plant_mask, (cx, cy), radius, 255, -1)
    
    # Inpaint to remove plants (replace with surrounding background)
    clean_background = cv2.inpaint(ref_img_rgb, plant_mask, 10, cv2.INPAINT_TELEA)
    
    # =========================================================
    # PLOT TRAJECTORIES
    # =========================================================
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Show clean background (plants removed)
    ax.imshow(clean_background)
    
    # Setup colors
    track_ids = list(unified_df['track_id'].dropna().unique())
    np.random.seed(42)
    bright_colors = [
        '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7',
        '#DDA0DD', '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9',
        '#F8B500', '#FF69B4', '#00CED1', '#32CD32', '#FF4500',
        '#9370DB', '#20B2AA', '#FFD700', '#FF1493', '#00FA9A',
    ]
    color_map = {tid: bright_colors[i % len(bright_colors)] for i, tid in enumerate(track_ids)}
    
    # Plot each track
    for track_id in track_ids:
        track_data = unified_df[unified_df['track_id'] == track_id].sort_values('frame')
        if len(track_data) < 2:
            continue
        
        color = color_map[track_id]
        xs = track_data['centroid_x'].values
        ys = track_data['centroid_y'].values
        
        # Draw trajectory line
        ax.plot(xs, ys, '-', linewidth=1.5, color=color, alpha=0.8)
        
        # Draw start point (small dot)
        ax.scatter(xs[0], ys[0], s=40, color=color, marker='o', 
                  edgecolor='white', linewidth=0.5, zorder=5)
        
        # Draw end point (larger dot with glow)
        ax.scatter(xs[-1], ys[-1], s=120, color=color, alpha=0.3, zorder=9)  # glow
        ax.scatter(xs[-1], ys[-1], s=80, color=color, 
                  edgecolor='white', linewidth=1, zorder=10)
    
    ax.set_title('Individual Trajectories Over Time', fontsize=VIZ_TITLE_FONTSIZE, fontweight='bold')
    ax.axis('off')
    
    plt.tight_layout()
    
    # Save
    fig_path = OUTPUT_DIR + "figure5_trajectories.png"
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved: {fig_path}")
    
    plt.show()
    plt.close('all')  # Ensure figure is closed
    
    print(f"\nPlotted {len(track_ids)} track trajectories")
    print("Plants removed from background using inpainting")

else:
    print("No tracking data found.")


## 5.6 Tracking Animation
Animated visualization with fading trails.

In [ ]:
# ============================================
# TRACKING ANIMATION WITH FADING TRAILS
# ============================================
# Same tracks as static (cell 48): background, colors, line style, dots, ID labels.
# Trails grow from frame 0 to current; fade along path (older→transparent, newer→static opacity).

from matplotlib.animation import FuncAnimation
import gc

if len(unified_df) > 0:
    print("Creating tracking animation...")
    
    FPS = 12
    all_frames = sorted(unified_df['frame'].unique())
    frame_step = 2
    animation_frames = all_frames[::frame_step][:200]
    
    # Background: exact logic from cell 48 (centroid + radius; bbox when present, else 30)
    print("Creating clean background...")
    # Load reference image - works for both video and images
    ref_img_rgb = get_reference_frame(0)
    plant_mask = np.zeros(ref_img_rgb.shape[:2], dtype=np.uint8)
    for frame_num in unified_df['frame'].unique():
        frame_data = unified_df[unified_df['frame'] == frame_num]
        for _, row in frame_data.iterrows():
            if pd.notna(row['centroid_x']) and pd.notna(row['centroid_y']):
                cx, cy = int(row['centroid_x']), int(row['centroid_y'])
                if 'bbox_minr' in row and pd.notna(row['bbox_minr']):
                    h = int(row['bbox_maxr'] - row['bbox_minr'])
                    w = int(row['bbox_maxc'] - row['bbox_minc'])
                    radius = max(h, w) // 2 + 5
                else:
                    radius = 30
                cv2.circle(plant_mask, (cx, cy), radius, 255, -1)
    clean_background = cv2.inpaint(ref_img_rgb, plant_mask, 10, cv2.INPAINT_TELEA)
    print("Background ready!")
    
    # Colors: same as static (cell 48) — 20 bright_colors, np.random.seed(42)
    track_ids = list(unified_df['track_id'].dropna().unique())
    np.random.seed(42)
    bright_colors = [
        '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7',
        '#DDA0DD', '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9',
        '#F8B500', '#FF69B4', '#00CED1', '#32CD32', '#FF4500',
        '#9370DB', '#20B2AA', '#FFD700', '#FF1493', '#00FA9A',
    ]
    color_map = {tid: bright_colors[i % len(bright_colors)] for i, tid in enumerate(track_ids)}
    
    fig, ax = plt.subplots(figsize=(14, 10))
    
    def animate(frame_idx):
        ax.clear()
        curr_frame = animation_frames[frame_idx]
        curr_hour = curr_frame * MINUTES_PER_FRAME / 60
        ax.imshow(clean_background)
        ax.axis('off')
        
        frame_data = unified_df[unified_df['frame'] == curr_frame]
        
        for tid in track_ids:
            hist = unified_df[(unified_df['track_id'] == tid) & (unified_df['frame'] <= curr_frame)].sort_values('frame')
            if len(hist) < 1:
                continue
            color = color_map[tid]
            xs, ys = hist['centroid_x'].values, hist['centroid_y'].values
            
            # Trail with fade: segment-wise alpha = 0.2 + 0.6*(i+1)/(len(xs)-1), linewidth=1.5
            if len(hist) >= 2:
                for i in range(len(xs) - 1):
                    alpha = 0.2 + 0.6 * (i + 1) / (len(xs) - 1)
                    ax.plot([xs[i], xs[i+1]], [ys[i], ys[i+1]], '-', linewidth=1.5, color=color, alpha=alpha)
            
            # Start dot (same as static)
            ax.scatter(xs[0], ys[0], s=40, color=color, marker='o', edgecolor='white', linewidth=0.5, zorder=5)
            
            # Current (end) dot: glow + main (same as static)
            ax.scatter(xs[-1], ys[-1], s=120, color=color, alpha=0.3, zorder=9)
            ax.scatter(xs[-1], ys[-1], s=80, color=color, edgecolor='white', linewidth=1, zorder=10)
            
            # ID label at current position
            ax.text(xs[-1] + 12, ys[-1] - 5, str(int(tid)), fontsize=7, fontweight='bold', color=color, zorder=11)
        
        ax.text(0.02, 0.98, f't = {curr_hour:.1f} h', transform=ax.transAxes, fontsize=14,
               fontweight='bold', color='black', va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        n_cells = len(frame_data['track_id'].dropna().unique())
        ax.text(0.98, 0.98, f'n = {n_cells}', transform=ax.transAxes, fontsize=14,
               fontweight='bold', color='black', va='top', ha='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        return []
    
    print(f"Generating {len(animation_frames)} frames...")
    anim = FuncAnimation(fig, animate, frames=len(animation_frames), interval=1000//FPS, blit=False, repeat=False)
    gif_path = OUTPUT_DIR + "tracking_animation_fading.gif"
    print(f"Saving to {gif_path}...")
    try:
        anim.save(gif_path, writer='pillow', fps=FPS)
        anim.event_source.stop()  # Stop the animation
        del anim  # Release the animation object
        # Make GIF non-looping (play once then stop)
        from PIL import Image, ImageSequence
        with Image.open(gif_path) as img:
            frames = [frame.copy() for frame in ImageSequence.Iterator(img)]
            if frames:
                frames[0].save(gif_path, save_all=True, append_images=frames[1:], duration=img.info.get('duration', 1000//FPS), loop=0)
        print(f"Saved: {gif_path}")
    except Exception as e:
        print(f"Error saving: {e}")
    finally:
        plt.close(fig)
        plt.close('all')
        gc.collect()
    from IPython.display import Image, display
    # Display removed - uncomment if you want to see it in notebook
    # display(Image(filename=gif_path))
    print("Done!")

else:
    print("No data found.")


In [ ]:
# ============================================
# BOUNDARY MODEL RESULTS GIF (RIGHT SIDE ONLY)
# ============================================
# Saves tracking overlay to OUTPUT_DIR/boundary_model_results.gif
# Crops to right half of the frame.

from matplotlib.animation import FuncAnimation
import gc

GIF_OUTPUT_PATH = OUTPUT_DIR + 'boundary_model_results.gif'
CROP_RIGHT_HALF = True  # Only show right side

if len(unified_df) > 0:
    print("Creating boundary model results GIF (right side only)...")
    
    FPS = 12
    all_frames = sorted(unified_df['frame'].unique())
    frame_step = 2
    animation_frames = all_frames[::frame_step][:200]
    
    print("Creating clean background...")
    ref_img_rgb = get_reference_frame(0)
    H, W = ref_img_rgb.shape[:2]
    x_offset = W // 2 if CROP_RIGHT_HALF else 0
    crop_w = W - x_offset if CROP_RIGHT_HALF else W
    
    plant_mask = np.zeros(ref_img_rgb.shape[:2], dtype=np.uint8)
    for frame_num in unified_df['frame'].unique():
        frame_data = unified_df[unified_df['frame'] == frame_num]
        for _, row in frame_data.iterrows():
            if pd.notna(row['centroid_x']) and pd.notna(row['centroid_y']):
                cx, cy = int(row['centroid_x']), int(row['centroid_y'])
                if 'bbox_minr' in row and pd.notna(row['bbox_minr']):
                    h = int(row['bbox_maxr'] - row['bbox_minr'])
                    w = int(row['bbox_maxc'] - row['bbox_minc'])
                    radius = max(h, w) // 2 + 5
                else:
                    radius = 30
                cv2.circle(plant_mask, (cx, cy), radius, 255, -1)
    clean_background_full = cv2.inpaint(ref_img_rgb, plant_mask, 10, cv2.INPAINT_TELEA)
    clean_background = clean_background_full[:, x_offset:x_offset + crop_w] if CROP_RIGHT_HALF else clean_background_full
    
    track_ids = list(unified_df['track_id'].dropna().unique())
    np.random.seed(42)
    bright_colors = [
        '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7',
        '#DDA0DD', '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9',
        '#F8B500', '#FF69B4', '#00CED1', '#32CD32', '#FF4500',
        '#9370DB', '#20B2AA', '#FFD700', '#FF1493', '#00FA9A',
    ]
    color_map = {tid: bright_colors[i % len(bright_colors)] for i, tid in enumerate(track_ids)}
    
    fig, ax = plt.subplots(figsize=(7, 10))
    
    def animate(frame_idx):
        ax.clear()
        curr_frame = animation_frames[frame_idx]
        curr_hour = curr_frame * MINUTES_PER_FRAME / 60
        ax.imshow(clean_background)
        ax.axis('off')
        
        frame_data = unified_df[unified_df['frame'] == curr_frame]
        
        for tid in track_ids:
            hist = unified_df[(unified_df['track_id'] == tid) & (unified_df['frame'] <= curr_frame)].sort_values('frame')
            if len(hist) < 1:
                continue
            color = color_map[tid]
            xs = hist['centroid_x'].values - x_offset
            ys = hist['centroid_y'].values
            mask = (xs >= 0) & (xs < crop_w)
            xs, ys = xs[mask], ys[mask]
            if len(xs) < 1:
                continue
            
            if len(xs) >= 2:
                for i in range(len(xs) - 1):
                    alpha = 0.2 + 0.6 * (i + 1) / max(1, len(xs) - 1)
                    ax.plot([xs[i], xs[i+1]], [ys[i], ys[i+1]], '-', linewidth=1.5, color=color, alpha=alpha)
            
            ax.scatter(xs[0], ys[0], s=40, color=color, marker='o', edgecolor='white', linewidth=0.5, zorder=5)
            ax.scatter(xs[-1], ys[-1], s=120, color=color, alpha=0.3, zorder=9)
            ax.scatter(xs[-1], ys[-1], s=80, color=color, edgecolor='white', linewidth=1, zorder=10)
            ax.text(xs[-1] + 12, ys[-1] - 5, str(int(tid)), fontsize=7, fontweight='bold', color=color, zorder=11)
        
        ax.text(0.02, 0.98, f't = {curr_hour:.1f} h', transform=ax.transAxes, fontsize=14,
               fontweight='bold', color='black', va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        n_cells = len(frame_data['track_id'].dropna().unique())
        ax.text(0.98, 0.98, f'n = {n_cells}', transform=ax.transAxes, fontsize=14,
               fontweight='bold', color='black', va='top', ha='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        return []
    
    print(f"Generating {len(animation_frames)} frames (right side only)...")
    anim = FuncAnimation(fig, animate, frames=len(animation_frames), interval=1000//FPS, blit=False, repeat=False)
    print(f"Saving to {GIF_OUTPUT_PATH}...")
    try:
        import os
        os.makedirs(os.path.dirname(GIF_OUTPUT_PATH), exist_ok=True)
        anim.save(GIF_OUTPUT_PATH, writer='pillow', fps=FPS)
        anim.event_source.stop()
        del anim
        from PIL import Image, ImageSequence
        with Image.open(GIF_OUTPUT_PATH) as img:
            frames = [frame.copy() for frame in ImageSequence.Iterator(img)]
            if frames:
                frames[0].save(GIF_OUTPUT_PATH, save_all=True, append_images=frames[1:], duration=img.info.get('duration', 1000//FPS), loop=0)
        print(f"Saved: {GIF_OUTPUT_PATH}")
    except Exception as e:
        print(f"Error saving: {e}")
    finally:
        plt.close(fig)
        plt.close('all')
        gc.collect()
else:
    print("No unified_df. Run tracking first.")


---
# PART 6: LINEAGE ANALYSIS
---
Analyze family relationships and generations.


## 6.2 Area by Generation
Size distribution across generations.

In [ ]:
# ============================================
# Plant Size Distribution by Generation - Amby-style (exact visualization)
# Vertical stacked histograms + red KDE + N, Mean, SD per generation
# ============================================
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
from matplotlib.ticker import FormatStrFormatter

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 18

gen_df = unified_df[unified_df['generation'].notna()].copy()
gen_df['generation'] = gen_df['generation'].astype(int)

if len(gen_df) > 0:
    generations = sorted(gen_df['generation'].unique())
    n_gens = len(generations)

    # Shared bins and xlim (Amby-style)
    gen0_data = gen_df[gen_df['generation'] == generations[0]]['area_mm2']
    _, bin_edges_0 = np.histogram(gen0_data, bins=20)
    bin_width = bin_edges_0[1] - bin_edges_0[0]
    max_val = gen_df['area_mm2'].max()
    common_bins = np.arange(0, max_val + bin_width, bin_width)
    global_xlim = (0, max_val + bin_width)

    fig, axes = plt.subplots(n_gens, 1, figsize=(8, 3 * n_gens), sharex=True)
    if n_gens == 1:
        axes = [axes]

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

    for i, g in enumerate(generations):
        ax = axes[i]
        gen_data = gen_df[gen_df['generation'] == g]['area_mm2'].values
        gen_data = gen_data[gen_data > 0]
        n_fronds = gen_df[gen_df['generation'] == g]['track_id'].nunique()

        if len(gen_data) == 0:
            ax.text(0.5, 0.5, "Gen {}\nNo data".format(g), ha='center', va='center', transform=ax.transAxes)
            ax.set_ylabel("Gen {}".format(g), fontsize=20, fontweight='bold')
            continue

        mean_a = np.mean(gen_data)
        sd_a = np.std(gen_data)

        ax.hist(gen_data, bins=common_bins, density=True, alpha=0.7,
                color=colors[i % len(colors)], edgecolor='black', linewidth=0.5)

        try:
            kde = gaussian_kde(gen_data)
            x_plot = np.linspace(0, max_val, 1000)
            ax.plot(x_plot, kde(x_plot), color='#d62728', linewidth=3)
        except Exception:
            pass

        ax.text(0.95, 0.85, "N = {} (fronds)".format(n_fronds), transform=ax.transAxes,
                fontsize=18, ha='right', va='top', fontweight='bold')
        stats_text = "Mean = {:.2f}\nSD = {:.2f}".format(mean_a, sd_a)
        props = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray')
        ax.text(0.70, 0.65, stats_text, transform=ax.transAxes, fontsize=14, fontweight='bold',
                verticalalignment='top', bbox=props)

        ax.set_ylabel("Gen {}".format(g), fontsize=20, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.grid(False)
        ax.tick_params(axis='both', labelsize=16, width=1.5, length=6, labelbottom=True)
        for lbl in ax.get_xticklabels() + ax.get_yticklabels():
            lbl.set_fontweight('normal')
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
        ax.set_xlim(global_xlim)
        ax.set_xlabel('Area (mm²)', fontsize=22, fontweight='bold')
    fig.suptitle('Plant Size Distribution by Generation', fontsize=24, fontweight='bold', y=1.02)
    plt.tight_layout()
    for ax in axes:
        for lbl in ax.get_xticklabels() + ax.get_yticklabels():
            lbl.set_fontweight('normal')
    
    fig_path = OUTPUT_DIR + "figure7_area_by_generation.png"
    try:
        plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
        print("Saved: {}".format(fig_path))
    except (TimeoutError, OSError) as e:
        import os
        local_dir = os.path.join(os.getcwd(), 'paper_analysis_output')
        os.makedirs(local_dir, exist_ok=True)
        fig_path = os.path.join(local_dir, "figure7_area_by_generation.png")
        plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
        print("Save failed; saved locally: {}".format(fig_path))
    plt.show()

    print("\n{}\nGeneration Summary\n{}".format("=" * 50, "=" * 50))
    for g in generations:
        gd = gen_df[gen_df['generation'] == g]['area_mm2']
        print("Generation {}: N={}, Mean={:.3f} mm², SD={:.3f} mm²".format(g, len(gd), gd.mean(), gd.std()))
    print("=" * 50)
else:
    print("No generation data available")


---
# PART 7: STATISTICAL ANALYSIS
---
Calculate quantitative metrics.


## 7.2 Cluster Variability
Compare variation between vs within clusters.

In [ ]:
# ============================================
# DELIVERABLE: Inter vs Intra-Cluster Variability
# ============================================

# Calculate doubling time for each cluster
cluster_doubling_times = []

for cluster_id in unified_df['cluster'].dropna().unique():
    cluster_data = unified_df[unified_df['cluster'] == cluster_id]
    cluster_pop = cluster_data.groupby('frame').size().reset_index(name='count')
    cluster_pop['hours'] = cluster_pop['frame'] * MINUTES_PER_FRAME / 60
    
    if len(cluster_pop) > 5 and cluster_pop['count'].iloc[-1] > cluster_pop['count'].iloc[0]:
        try:
            popt, _ = curve_fit(exponential_growth, 
                               cluster_pop['hours'].values, 
                               cluster_pop['count'].values,
                               p0=[cluster_pop['count'].iloc[0], 0.01],
                               maxfev=5000)
            r = popt[1]
            if r > 0:
                dt = np.log(2) / r
                cluster_doubling_times.append({
                    'cluster': cluster_id,
                    'doubling_time': dt,
                    'start_pop': cluster_pop['count'].iloc[0],
                    'end_pop': cluster_pop['count'].iloc[-1]
                })
        except:
            pass

if len(cluster_doubling_times) > 0:
    dt_df = pd.DataFrame(cluster_doubling_times)
    
    print("="*50)
    print("CLUSTER VARIABILITY ANALYSIS")
    print("="*50)
    print(f"\nClusters analyzed: {len(dt_df)}")
    print(f"\nDoubling time statistics:")
    print(f"  Mean: {dt_df['doubling_time'].mean():.1f} hours")
    print(f"  Std:  {dt_df['doubling_time'].std():.1f} hours")
    print(f"  Min:  {dt_df['doubling_time'].min():.1f} hours")
    print(f"  Max:  {dt_df['doubling_time'].max():.1f} hours")
    print(f"\nCoefficient of Variation: {dt_df['doubling_time'].std()/dt_df['doubling_time'].mean()*100:.1f}%")
    print("="*50)
    
    # Plot distribution
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(dt_df['doubling_time'], bins=10, edgecolor='black', alpha=0.7, color='#2E86AB')
    ax.axvline(dt_df['doubling_time'].mean(), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {dt_df["doubling_time"].mean():.1f}h')
    ax.set_xlabel('Doubling Time (hours)', fontsize=14)
    ax.set_ylabel('Number of Clusters', fontsize=14)
    ax.set_title('Doubling Time Distribution Across Clusters', fontsize=16, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    fig_path = OUTPUT_DIR + "figure9_cluster_variability.png"
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved: {fig_path}")
    
    plt.show()
else:
    print("Not enough clusters for variability analysis")

---
# PART 8: COMPARISON
---
Compare petri dish vs microfluidics results.

## 8.1 Dataset Comparison
Compare growth dynamics between datasets.

In [ ]:
# ============================================
# DELIVERABLE: Petri Dish vs Linear Trench Comparison
# ============================================
# NOTE: This requires Linear Trench data to be processed first
# Update LINEAR_TRENCH_DATA path once available

LINEAR_TRENCH_DATA = None  # Path to linear trench unified_tracking_data.csv

if LINEAR_TRENCH_DATA and Path(LINEAR_TRENCH_DATA).exists():
    # Load linear trench data
    trench_df = pd.read_csv(LINEAR_TRENCH_DATA)
    trench_pop = trench_df.groupby('frame').size().reset_index(name='count')
    trench_pop['hours'] = trench_pop['frame'] * MINUTES_PER_FRAME / 60
    
    # Create comparison plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Petri dish (this notebook's data)
    ax.step(pop_data['hours'], pop_data['count'], where='post', 
            linewidth=2, color='#2E86AB', label='Petri Dish (unlimited)', alpha=0.8)
    
    # Linear trench
    ax.step(trench_pop['hours'], trench_pop['count'], where='post', 
            linewidth=2, color='#E94F37', label='Linear Trench (constrained)', alpha=0.8)
    
    ax.set_xlabel('Time (hours)', fontsize=14)
    ax.set_ylabel('Number of Individuals', fontsize=14)
    ax.set_title('Population Dynamics: Unlimited vs Constrained Growth', 
                 fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    
    plt.tight_layout()
    
    fig_path = OUTPUT_DIR + "figure10_petri_vs_trench.png"
    plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved: {fig_path}")
    
    plt.show()
else:
    print("Linear Trench data not available yet.")
    print("To generate comparison figure:")
    print("1. Process linear trench images through the same pipeline")
    print("2. Set LINEAR_TRENCH_DATA path above")
    print("3. Re-run this cell")

---
# PART 9: SUMMARY
---

## 9.1 Generated Outputs
List all figures and data files.

In [ ]:
# ============================================
# SUMMARY: List All Generated Outputs
# ============================================

print("="*60)
print("PAPER ANALYSIS COMPLETE")
print("="*60)
print(f"\nOutput directory: {OUTPUT_DIR}")
print("\n" + "-"*60)
print("DATA FILES:")
print("-"*60)
print("  • unified_tracking_data.csv - Master tracking dataset")
print("  • lineage_data.csv - Parent-child relationships")
print("  • track_data.csv - Raw track coordinates")

print("\n" + "-"*60)
print("FIGURES:")
print("-"*60)
print("  • figure1_segmentation_demo.png - Raw vs Segmented")
print("  • figure2_population_step_function.png - Population over time")
print("  • figure3_single_cluster_crops.png - Cluster growth with images")
print("  • figure4_single_cluster_population.png - Single cluster pop")
print("  • figure5_trajectories.png - All track trajectories")
print("  • figure6_lineage_tree.png - Family tree diagram")
print("  • figure7_area_by_generation.png - Size by generation")
print("  • figure8_doubling_time.png - Exponential fit")
print("  • figure9_cluster_variability.png - Inter-cluster variation")
print("  • figure10_petri_vs_trench.png - Comparison (needs trench data)")

print("\n" + "-"*60)
print("KEY METRICS:")
print("-"*60)
print(f"  • Total frames processed: {len(image_files)}")
print(f"  • Total tracks: {unified_df['track_id'].nunique()}")
print(f"  • Budding events: {len(lineage_df)}")
print(f"  • Max generation: {unified_df['generation'].max()}")
if 'doubling_time' in dir():
    print(f"  • Doubling time: {doubling_time:.1f} hours")
print("="*60)

In [ ]:
# ============================================
# SAVE TRACKING DATA
# ============================================

# Save unified tracking data
unified_path = OUTPUT_DIR + "unified_tracking_data.csv"
unified_df.to_csv(unified_path, index=False)
print(f"Saved: {unified_path}")

# Save lineage data
lineage_path = OUTPUT_DIR + "lineage_data.csv"
lineage_df.to_csv(lineage_path, index=False)
print(f"Saved: {lineage_path}")

# Save track data
track_path = OUTPUT_DIR + "track_data.csv"
track_df.to_csv(track_path, index=False)
print(f"Saved: {track_path}")

print("\nAll data saved! You can load these files to skip the processing steps.")

In [ ]:
# Show first frame to identify wells
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# IMAGE_DIRECTORY already set in config cell above
image_files = sorted(Path(IMAGE_DIRECTORY).glob('*.jpeg'))

# Load first image - works for both video and images
img_rgb = get_reference_frame(0)

# Show image
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(img_rgb)
ax.set_title('Frame 0 - Which well do you want to focus on?', fontsize=14)

plt.tight_layout()
plt.show()

print(f"Image size: {img_rgb.shape[1]} x {img_rgb.shape[0]} pixels")

In [ ]:
# Let's see the full image with the tracking data overlaid to help you pick a region
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# IMAGE_DIRECTORY already set in config cell above
image_files = sorted(Path(IMAGE_DIRECTORY).glob('*.jpeg'))

# Load first image - works for both video and images
img_rgb = get_reference_frame(0)
h, w = img_rgb.shape[:2]

# Show image with coordinate grid
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(img_rgb)

# Add coordinate ticks
ax.set_xticks(np.arange(0, w, 200))
ax.set_yticks(np.arange(0, h, 200))
ax.grid(True, alpha=0.3, color='yellow')

ax.set_title(f'Frame 0 | Image size: {w} x {h} pixels\nTell me the ROI coordinates', fontsize=12)
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')

plt.tight_layout()
plt.show()

print(f"\nImage dimensions: {w} x {h} pixels")
print("\nTo select a region, give me approximate coordinates:")
print("  - x_min, x_max (left to right)")
print("  - y_min, y_max (top to bottom)")
print("\nOr say 'full image' if the whole image is one dish")

In [ ]:
import sys
print(sys.executable)
print(sys.version)